# Think on the Graph: every output of the run MY29


## 0. Input
Program:
```java
double a = 0.1 + 0.2;
if (a == 0.3) {
    System.out.println("equal");
} else {
    System.out.println("not equal");
}
```
Question: What is printed? Explain what the (double) does here.

Student's answer: 3.5. The cast makes it a double division, so you get the decimal part.
Settings: width 3, depth 3, prune llm, backend claude-cli, model claude-opus-5-5


## 1. Stage 1: the subgraph G from the code (no model)
Masked code (string and char literals and comments blanked; this is what the patterns see):
```
double a = 0.1 + 0.2;
if (a == 0.3) {
    System.out.println("     ");
} else {
    System.out.println("         ");
}
```
Which code text tagged which concept (a generic concept fires on almost any Java line; it starts the walk only if the answer repeats its tag):
| concept | generic | channel | matched text | line |
|---|---|---|---|---|
| arithmetic_operators | False | pattern | '1 + 0' | 1 |
| assignment | True | pattern | '=' | 1 |
| boolean_expression | False | pattern | 'if (' | 2 |
| comparison_operators | False | pattern | 'if (a ==' | 2 |
| control_flow | False | pattern | 'if (' | 2 |
| double | True | pattern | 'double' | 1 |
| double | True | pattern | '0.1' | 1 |
| double | True | type | 'double' | 1 |
| expression | True | pattern | '1 +' | 1 |
| expression | True | pattern | '= 0.1 + 0.2;' | 1 |
| floating_point_precision | False | pattern | '0.1 + 0.2' | 1 |
| floating_point_precision | False | pattern | '== 0.3' | 2 |
| floating_point_precision | False | signal | 'double a = 0.1 + 0.2;\nif (a ==' | 1 |
| if_else_statement | False | pattern | 'else' | 4 |
| if_statement | False | pattern | 'if (' | 2 |
| println | False | pattern | 'System.out.println(' | 3 |
| statement | True | pattern | ';' | 1 |
| string_literal | True | pattern | '"equal"' | 3 |

G: 13 concept nodes (5 generic), 11 concept edges, 99 misconception / error edges, 76 misconception / error nodes, 65 level nodes, 102 level-misconception links, 2 connected piece(s).
Note: concept nodes form 2 pieces (the KG has no edge between them): arithmetic_operators,assignment,boolean_expression,comparison_operators,control_flow,expression,if_else_statement,if_statement,println,statement,string_literal | double,floating_point_precision

Concept edges of G (the links the walk can read; `pre:` = prerequisite A -> B, `par:` = parallel):
| edge | type | rule (logic) |
|---|---|---|
| par:assignment~comparison_operators | parallel | x = y stores the value of y in x (a statement with a side effect); x == y only tests whether the two values are equal and gives a boolean. Writing = where == is meant is a compile error with int operands and a silent bug with boolean operands. |
| pre:arithmetic_operators->expression | prerequisite | An operator combines operand values a and b into a new value, so operators build larger expressions from smaller ones. |
| pre:boolean_expression->if_statement | prerequisite | An if runs its block only when its boolean expression evaluates to true. |
| pre:comparison_operators->boolean_expression | prerequisite | A comparison such as x < y produces a boolean result; comparisons together with boolean logic provide the basis for decision-making. |
| pre:control_flow->if_statement | prerequisite | An if changes the default top-to-bottom flow by running a block only when its condition is true. |
| pre:double->floating_point_precision | prerequisite | A double approximates real numbers, so its arithmetic is not exact. |
| pre:expression->comparison_operators | prerequisite | Comparison operators build expressions whose value is a boolean, so one must already know what an expression is to understand a < b. |
| pre:if_statement->if_else_statement | prerequisite | if-else extends if with a second block that runs exactly when the condition is false. |
| pre:statement->control_flow | prerequisite | Control flow is the order in which statements run. |
| pre:statement->println | prerequisite | System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence. |
| pre:string_literal->println | prerequisite | println("s") outputs exactly the characters between the quotes and then ends the line. |

Misconception / error edges per concept (count), and level ways per concept (count):
| concept | misconception / error edges | level ways (L1-L4 + L0 non-misconception) |
|---|---|---|
| arithmetic_operators | 3 | 18 |
| assignment | 15 | 24 |
| boolean_expression | 8 | 19 |
| comparison_operators | 8 | 20 |
| control_flow | 11 | 15 |
| double | 2 | 19 |
| expression | 8 | 19 |
| floating_point_precision | 1 | 14 |
| if_else_statement | 6 | 17 |
| if_statement | 22 | 19 |
| println | 6 | 17 |
| statement | 5 | 19 |
| string_literal | 4 | 16 |

**Figure 1**

![Figure 1](figure01.png)
**Figure 2**

![Figure 2](figure02.png)


## 2. Clauses and start concepts (no model)
The answer split into clauses (at sentence ends, ';', and before 'because' / 'so'; quotes removed):
| clause | text |
|---|---|
| c1 | 3.5 |
| c2 | The cast makes it a double division |
| c3 | so you get the decimal part |

start: generic concepts whose tagging text occurs in the answer: double

start: BM25 of the answer against each tagged concept: double 11.19 (hits 3), arithmetic_operators 6.38 (hits 1), if_statement 6.16 (hits 1), println 5.40 (hits 1), boolean_expression 4.54 (hits 1), floating_point_precision 3.23 (hits 3), control_flow 2.61 (hits 1), comparison_operators 1.98 (hits 1), if_else_statement 1.71 (hits 1)

start: chosen double, arithmetic_operators, if_statement

start concepts (3): double, arithmetic_operators, if_statement

Start concepts: double, arithmetic_operators, if_statement

**Figure 3**

![Figure 3](figure03.png)


## 3. The walk, depth by depth
At each depth, for each current concept: the edges offered (a concept entering as a start concept offers its misconception / error edges and its level ways, as two families; otherwise it offers its concept edges), the prune prompt and the model's reply, then the label prompt(s) and the model's reply, parsed into label / clauses / confidence. Score = prune score x confidence.

### Depth 1: current concepts 'double', 'arithmetic_operators', 'if_statement'

#### double, misconceptions: 2 edges offered
Edges offered:
| edge |
|---|
| mis:double->FloatingPointIsExact \| wrong rule: Math.sqrt(2) * Math.sqrt(2) == 2 is true. \| right rule: A double is an approximation, so the product is close to 2 but not equal to 2. |
| mis:double->WholeNumberLiteralIsInt \| wrong rule: 207. and 2.3e4 are int values because they are whole numbers. \| right rule: A decimal point or scientific notation makes the literal a double. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{mis:double->WholeNumberLiteralIsInt (Score: 0.6)}
{mis:double->FloatingPointIsExact (Score: 0.4)}
```
Kept by pruning: mis:double->WholeNumberLiteralIsInt (0.60), mis:double->FloatingPointIsExact (0.40)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:double->WholeNumberLiteralIsInt \| wrong rule: 207. and 2.3e4 are int values because they are whole numbers. \| right rule: A decimal point or scientific notation makes the literal a double. |
| mis:double->FloatingPointIsExact \| wrong rule: Math.sqrt(2) * Math.sqrt(2) == 2 is true. \| right rule: A double is an approximation, so the product is close to 2 but not equal to 2. |
Model's reply:
```
mis:double->WholeNumberLiteralIsInt | not applied | clauses: none | confidence: 0.9
mis:double->FloatingPointIsExact | not applied | clauses: none | confidence: 0.8
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:double->WholeNumberLiteralIsInt | not applied |  |  | 0.0 |  |  |
| mis:double->FloatingPointIsExact | not applied |  |  | 0.0 |  |  |

#### double, level ways: 19 edges offered
Edges offered:
| edge |
|---|
| lvl:double@L0#0 \| level L0 way everyday_reading: Folk meaning: a number with a decimal point is the exact real number of school maths, and a whole-looking number such as 207. or 2.3e4 is an integer. Java rule: a double is a 64-bit binary approximation from a finite set of values, and any literal with a decimal point or exponent is a double (207. is 207.0, 2.3e4 is 23000.0). |
| lvl:double@L0#3 \| level L0 way inconsistent_trace: Points to LT (at each operator the narrower operand is promoted, then the operation is done in double and rounded). Applied unevenly: the student promotes to double on one line but keeps integer division on another, e.g. treats 1/2 * x as 0.5 * x on one pass and as 0 on the next. (here mixing in: "Math.sqrt(2) * Math.sqrt(2) == 2 is true") |
| lvl:double@L1#0 \| level L1 way names_the_element: double is a primitive data type: Java's 64-bit IEEE 754 floating-point type for representing real numbers. |
| lvl:double@L1#1 \| level L1 way states_the_rule: Values are 64-bit IEEE 754 numbers (about 15 significant digits) plus the special values Infinity and NaN; +, -, *, / (and %) are defined; if at least one operand of a binary arithmetic operator is double, the operation is a floating-point operation. |
| lvl:double@L2#0 \| level L2 way line_by_line_trace: Operators are applied in precedence order, left operand first; at each operator, if one operand is double the other is widened to double before the operation, and the result is rounded to the nearest double before the next operator uses it. |
| lvl:double@L2#1 \| level L2 way output_prediction: The result has type double: 5.0/3.0 is 1.6666666666666667, 10/3.0 is 3.3333333333333335, 1.0/0.0 is Infinity, Math.sqrt(-1.0) is NaN; println always shows a decimal part (3.0, not 3). |
| lvl:double@L2#2 \| level L2 way one_step_execution: An assignment to a double variable stores the nearest representable value (x = x + 0.1 replaces x with the rounded sum); no double operation throws an exception, so the variable may end up holding Infinity or NaN. |
| lvl:double@L3#0 \| level L3 way relation_to_neighbour: Type flow decides the division: with two int operands a / b discards the fraction (10/3 is 3, and double d = 10/3; stores 3.0); if either operand is double the other is promoted and the fraction is kept (10/3.0 is 3.3333333333333335). |
| lvl:double@L3#1 \| level L3 way relation_to_neighbour: double and boolean never convert into each other (no cast is allowed either way); they meet only through comparisons, which take doubles and produce a boolean, e.g. (x < 3.1). |
| lvl:double@L3#2 \| level L3 way relation_to_neighbour: char is an integral numeric type, so a char widens automatically to double in arithmetic ('a' + 0.5 is 97.5), but a double needs an explicit cast to become a char. |
| lvl:double@L3#3 \| level L3 way relation_to_neighbour: Every double operation rounds its result to the nearest representable value, so double results carry rounding error that can accumulate; this is why == on computed doubles is unreliable. |
| lvl:double@L3#4 \| level L3 way relation_to_neighbour: For double operands a zero divisor is not an error: 1.0/0.0 is Infinity, -1.0/0.0 is -Infinity and 0.0/0.0 is NaN, whereas int division by zero throws an exception. |
| lvl:double@L3#5 \| level L3 way relation_to_neighbour: double is one of the eight primitive types and the default type of decimal literals (float would need an f suffix and is not used in the book). |
| lvl:double@L3#6 \| level L3 way purpose_of_the_whole: Used for measured or computed real quantities (physics formulas, roots, averages, probabilities); a double variable often plays the gatherer role, as in Harmonic's sum += 1.0/i. |
| lvl:double@L4#0 \| level L4 way predict_a_change: Changing a variable from int to double changes / from truncating to real division and changes the printed form (3 becomes 3.0); changing back needs a cast and loses the fraction. Invariant: values are approximations, so reordering a sum ((a+b)+c vs a+(b+c)) may change the last digits. |
| lvl:double@L4#1 \| level L4 way transfer_to_a_new_program: Use double for real-valued quantities where approximate results are acceptable (science, geometry, probabilities, averages); do not use it for counts, array indices, loop counters or money that must be exact (use int or long, e.g. cents). |
| lvl:double@L4#2 \| level L4 way diagnose_an_error: rule broken: A double cannot be assigned to an int (narrowing) without a cast. Kind: compile-time. Trigger class: A double expression assigned or passed where an int is required, e.g. int n = Math.sqrt(x); or int n = 2.0 * k; (possible lossy conversion). |
| lvl:double@L4#3 \| level L4 way diagnose_an_error: rule broken: Floating-point division needs at least one double operand. Kind: silent logic error. Trigger class: A formula assigned to a double whose division has only int operands, e.g. double y = 1 / 3; (gives 0.0) or (9/5)*c + 32. |
| lvl:double@L4#4 \| level L4 way diagnose_an_error: rule broken: Computed doubles are approximations and must not be compared with ==. Kind: silent logic error. Trigger class: == or != between doubles produced by arithmetic on non-dyadic fractions (0.1, 1/3, square roots). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:double@L3#0 (Score: 0.5)}
{lvl:double@L1#1 (Score: 0.3)}
{lvl:double@L4#3 (Score: 0.2)}
```
Kept by pruning: lvl:double@L3#0 (0.50), lvl:double@L1#1 (0.30), lvl:double@L4#3 (0.20)

Label prompt for level way lvl:double@L3#0 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> Type flow decides the division: with two int operands a / b discards the fraction (10/3 is 3, and double d = 10/3; stores 3.0); if either operand is double the other is promoted and the fraction is kept (10/3.0 is 3.3333333333333335).

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.7
```

Label prompt for level way lvl:double@L1#1 (the model saw only the concept, the way's name 'states the rule' and its text; no level):
> Values are 64-bit IEEE 754 numbers (about 15 significant digits) plus the special values Infinity and NaN; +, -, *, / (and %) are defined; if at least one operand of a binary arithmetic operator is double, the operation is a floating-point operation.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.7
```

Label prompt for level way lvl:double@L4#3 (the model saw only the concept, the way's name 'diagnose an error' and its text; no level):
> rule broken: Floating-point division needs at least one double operand. Kind: silent logic error. Trigger class: A formula assigned to a double whose division has only int operands, e.g. double y = 1 / 3; (gives 0.0) or (9/5)*c + 32.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.6
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:double@L3#0 | shown | 0.5 | 0.7 | 0.35 | c2, c3 | The cast makes it a double division; so you get the decimal part |
| lvl:double@L1#1 | shown | 0.3 | 0.7 | 0.21 | c2, c3 | The cast makes it a double division; so you get the decimal part |
| lvl:double@L4#3 | shown | 0.2 | 0.6 | 0.12 | c2, c3 | The cast makes it a double division; so you get the decimal part |

#### arithmetic_operators, misconceptions: 3 edges offered
Edges offered:
| edge |
|---|
| mis:arithmetic_operators->ArithmeticOnNumericStrings \| wrong rule: "10" - 1 or "6" * "2" works because the strings look like numbers. \| right rule: Arithmetic operators other than + are not defined for Strings; a String of digits must first be converted to a number, e.g. with Integer.parseInt. |
| mis:arithmetic_operators->CaretIsPower \| wrong rule: a^2 computes a squared. \| right rule: Java has no exponent operator; ^ is exclusive or, and a * a or Math.pow(a, 2) computes a squared. |
| mis:arithmetic_operators->RemainderIsSubtraction \| wrong rule: 9 % 5 means 9 minus 5 (a % b subtracts b from a). \| right rule: % gives the remainder after integer division: 9 % 5 is 4 because 9 = 1 * 5 + 4, and 17 % 5 is 2, not 12. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```

```
Kept by pruning: 

Parsed labels:
(none)

#### arithmetic_operators, level ways: 18 edges offered
Edges offered:
| edge |
|---|
| lvl:arithmetic_operators@L0#0 \| level L0 way everyday_reading: In maths notation, multiplication can be implicit (2x), ^ or a superscript means a power, ÷ is exact division, and % means percent. In Java * must be written, / depends on the operand types, % is the remainder, and there is no power operator (^ is exclusive or). |
| lvl:arithmetic_operators@L0#2 \| level L0 way inconsistent_trace: Points to LT (evaluate operands, then apply the operator in the promoted type). Applied unevenly, a student uses integer division for one / and real division for another / in the same int expression. (here mixing in: "a^2 computes a squared") |
| lvl:arithmetic_operators@L1#0 \| level L1 way names_the_element: Arithmetic operators: the binary operators + - * / % that combine two numeric operands into a numeric result. |
| lvl:arithmetic_operators@L1#1 \| level L1 way states_the_rule: Both operands must be numeric, and the result has the promoted type (int, long or double). * / % bind tighter than + -, all group left to right, and int / or % by zero throws ArithmeticException. Unary minus negates its operand (-x) and unary plus leaves it unchanged; they bind more tightly than * and /, so -a * b is (-a) * b. |
| lvl:arithmetic_operators@L2#0 \| level L2 way line_by_line_trace: The left operand is evaluated fully, then the right operand, and then the operator is applied in the promoted type. |
| lvl:arithmetic_operators@L2#1 \| level L2 way output_prediction: With ints, 17 / 3 is 5 and 17 % 3 is 2. With doubles, 5.0 / 3.0 is 1.6666666666666667. With mixed operands, the int is promoted first (10 / 3.0 is 3.333...). |
| lvl:arithmetic_operators@L2#2 \| level L2 way one_step_execution: An arithmetic operator never changes its operands. Each variable is replaced by its value, and only an assignment (or ++ or op=) stores the result. |
| lvl:arithmetic_operators@L3#0 \| level L3 way relation_to_neighbour: Operators build larger expressions from smaller ones, and each operand can itself be any expression. |
| lvl:arithmetic_operators@L3#1 \| level L3 way relation_to_neighbour: For ints, / gives the truncated quotient and % the remainder, so (a / b) * b + a % b == a. |
| lvl:arithmetic_operators@L3#2 \| level L3 way relation_to_neighbour: int / or % by zero throws a run-time exception, while floating-point division by zero gives Infinity or NaN. |
| lvl:arithmetic_operators@L3#3 \| level L3 way relation_to_neighbour: + adds when both operands are numbers and concatenates when either operand is a String, converting the other operand to a String. It is left-associative, so 1 + 2 + "3" is "33" but "1" + 2 + 3 is "123". |
| lvl:arithmetic_operators@L3#4 \| level L3 way purpose_of_the_whole: Arithmetic operators turn formulas into code: totals, averages, unit conversions (inch * CM_PER_INCH), and digit extraction (n % 10, n / 10). |
| lvl:arithmetic_operators@L4#0 \| level L4 way predict_a_change: Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power. |
| lvl:arithmetic_operators@L4#1 \| level L4 way transfer_to_a_new_program: Use the operators for arithmetic on numeric types. Use the Math library for powers, roots and absolute values, and parse numeric Strings before computing with them. |
| lvl:arithmetic_operators@L4#2 \| level L4 way diagnose_an_error: rule broken: int / and % by zero are undefined. Kind: run-time exception. Trigger class: A divisor that evaluates to 0 (empty count, zero input): ArithmeticException: / by zero. |
| lvl:arithmetic_operators@L4#3 \| level L4 way diagnose_an_error: rule broken: Arithmetic operators other than + are not defined for Strings. Kind: compile-time. Trigger class: "10" - 1 or "6" * "2": 'bad operand types for binary operator'. |
| lvl:arithmetic_operators@L4#4 \| level L4 way diagnose_an_error: rule broken: ^ is not exponentiation. Kind: silent logic error. Trigger class: Writing x^2 or 10^6 for a power (10^6 is 12). |
| lvl:arithmetic_operators@L4#5 \| level L4 way diagnose_an_error: rule broken: int arithmetic wraps silently on overflow. Kind: silent logic error. Trigger class: Results outside the int range. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:arithmetic_operators@L4#0 (Score: 0.45)}
{lvl:arithmetic_operators@L2#1 (Score: 0.35)}
{lvl:arithmetic_operators@L1#1 (Score: 0.2)}
```
Kept by pruning: lvl:arithmetic_operators@L4#0 (0.45), lvl:arithmetic_operators@L2#1 (0.35), lvl:arithmetic_operators@L1#1 (0.20)

Label prompt for level way lvl:arithmetic_operators@L4#0 (the model saw only the concept, the way's name 'predict a change' and its text; no level):
> Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.75
```

Label prompt for level way lvl:arithmetic_operators@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> With ints, 17 / 3 is 5 and 17 % 3 is 2. With doubles, 5.0 / 3.0 is 1.6666666666666667. With mixed operands, the int is promoted first (10 / 3.0 is 3.333...).

Model's reply:
```
Label: shown
Clauses: 1, 2, 3
Confidence: 0.6
```

Label prompt for level way lvl:arithmetic_operators@L1#1 (the model saw only the concept, the way's name 'states the rule' and its text; no level):
> Both operands must be numeric, and the result has the promoted type (int, long or double). * / % bind tighter than + -, all group left to right, and int / or % by zero throws ArithmeticException. Unary minus negates its operand (-x) and unary plus leaves it unchanged; they bind more tightly than * and /, so -a * b is (-a) * b.

Model's reply:
```
Label: shown
Clauses: 2, 3
Confidence: 0.55
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:arithmetic_operators@L4#0 | shown | 0.45 | 0.75 | 0.34 | c2, c3 | The cast makes it a double division; so you get the decimal part |
| lvl:arithmetic_operators@L2#1 | shown | 0.35 | 0.6 | 0.21 | c1, c2, c3 | 3.5; The cast makes it a double division; so you get the decimal part |
| lvl:arithmetic_operators@L1#1 | shown | 0.2 | 0.55 | 0.11 | c2, c3 | The cast makes it a double division; so you get the decimal part |

#### if_statement, misconceptions: 22 edges offered
Edges offered:
| edge |
|---|
| err:if_statement->BracelessMultiStatementBody \| wrong rule: Indenting several statements under an if or for makes them all part of its body, even without braces. \| right rule: Without braces only the first statement belongs to the if/for; the rest run once, after it. |
| err:if_statement->CurlyIfCondition \| wrong rule: Writes the if condition inside { } instead of ( ). \| right rule: The condition of an if is written in parentheses. |
| err:if_statement->SemicolonAfterIfHeader \| wrong rule: if (x > 0); System.out.println("positive"); prints only for positive x. \| right rule: The ; ends the if with an empty statement, so the next line always runs. |
| mis:if_statement->AssignCompares \| wrong rule: = compares two values instead of assigning \| right rule: x = y copies the value of y into x; x == y is what compares them. |
| mis:if_statement->BranchChosenByIntent \| wrong rule: Which branch runs is decided by what the programmer meant or by code outside the if, not by whether the condition is true. \| right rule: The then branch runs exactly when the condition evaluates to true at the moment it is tested; otherwise the else branch (if any) runs. |
| mis:if_statement->CodeAfterIfSkipped \| wrong rule: When the then branch runs, the code after the if is skipped. \| right rule: After the if finishes, execution continues with the next statement either way. |
| mis:if_statement->FalseConditionEndsProgram \| wrong rule: A false condition with no else ends the program. \| right rule: When an if condition is false and there is no else, execution continues with the statement after the if. |
| mis:if_statement->FalseConditionRestartsProgram \| wrong rule: When an if condition is false (and there is no else), control jumps back to the start of the program. \| right rule: A false condition skips the then branch and execution continues with the statement after the if statement. |
| mis:if_statement->IfConditionCheckedInHindsight \| wrong rule: Whether the branch runs is decided by the condition after the branch, or from the point inside the body where the condition becomes true. \| right rule: The condition is tested once, before the branch; the whole branch runs if it was true at that moment and none of it runs otherwise. |
| mis:if_statement->IfConditionWatched \| wrong rule: An if statement watches its condition while the program runs and fires as soon as the condition becomes true (if = whenever). \| right rule: An if condition is tested once, when control reaches the if statement; later changes to the variables do not trigger the branch. |
| mis:if_statement->IfInLoopUnchecked \| wrong rule: An if inside a loop body runs its branch on every pass without testing its condition. \| right rule: The if condition is tested again on every pass, with the current values, so the branch runs only on passes where it is true. |
| mis:if_statement->IfIsLoop \| wrong rule: The if body keeps repeating as long as its condition stays true. \| right rule: An if body runs at most once; only a loop repeats its body. |
| mis:if_statement->IfRequiresElse \| wrong rule: Every if statement requires an else. \| right rule: An if statement may stand alone; the else part is optional. |
| mis:if_statement->IfWaitsForCondition \| wrong rule: An IF statement earlier in the program stays active and fires later, e.g. inside a later loop, when its condition becomes true. \| right rule: An if is evaluated once when control reaches it; afterwards it is inactive unless control returns to it. |
| mis:if_statement->LoopForOneTimeTest \| wrong rule: Uses a loop where a condition needs to be tested only once. \| right rule: A test made once is an if; a loop repeats. |
| mis:if_statement->MapToBooleanWithIf \| wrong rule: you need an if to turn a boolean expression into a boolean \| right rule: ok = (x > y); stores the result of the comparison directly. |
| mis:if_statement->MissingBraces \| wrong rule: Omits braces around a single-statement body of if, for or while. \| right rule: Braces are optional for a single-statement body, but using them anyway avoids bugs when statements are added later. |
| mis:if_statement->NextStatementIsElse \| wrong rule: In if (c) A; B; the statement B runs only when c is false. \| right rule: B is not part of the if; it runs every time. |
| mis:if_statement->ProgramLooksAhead \| wrong rule: The program knows or sees what later lines will do and acts on those goals at the current line, e.g. it stops 'at ten' because it wants to draw a box. \| right rule: Each statement executes using only the current state; later lines have no effect until control reaches them. |
| mis:if_statement->SequentialIfsAreNested \| wrong rule: Of two if statements one after the other, the second runs its body only if both conditions are true. \| right rule: Two ifs in sequence are independent: each tests only its own condition. |
| mis:if_statement->StatementAfterIfInBranch \| wrong rule: A statement placed after the if statement belongs to its branch, so it runs only when the condition is true. \| right rule: The branch ends at its closing brace (or after its single statement); the next statement runs whatever the condition was. |
| mis:if_statement->ThenAlwaysExecutes \| wrong rule: The then branch always runs. \| right rule: The then branch runs only when the condition is true. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:if_statement->BranchChosenByIntent (0.50), mis:if_statement->ThenAlwaysExecutes (0.30), mis:if_statement->ProgramLooksAhead (0.20)
```
Kept by pruning: mis:if_statement->BranchChosenByIntent (0.50), mis:if_statement->ThenAlwaysExecutes (0.30), mis:if_statement->ProgramLooksAhead (0.20)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:if_statement->BranchChosenByIntent \| wrong rule: Which branch runs is decided by what the programmer meant or by code outside the if, not by whether the condition is true. \| right rule: The then branch runs exactly when the condition evaluates to true at the moment it is tested; otherwise the else branch (if any) runs. |
| mis:if_statement->ThenAlwaysExecutes \| wrong rule: The then branch always runs. \| right rule: The then branch runs only when the condition is true. |
| mis:if_statement->ProgramLooksAhead \| wrong rule: The program knows or sees what later lines will do and acts on those goals at the current line, e.g. it stops 'at ten' because it wants to draw a box. \| right rule: Each statement executes using only the current state; later lines have no effect until control reaches them. |
Model's reply:
```
mis:if_statement->BranchChosenByIntent | not applied | clauses: none | confidence: 0.85
mis:if_statement->ThenAlwaysExecutes | not applied | clauses: none | confidence: 0.85
mis:if_statement->ProgramLooksAhead | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:if_statement->BranchChosenByIntent | not applied |  |  | 0.0 |  |  |
| mis:if_statement->ThenAlwaysExecutes | not applied |  |  | 0.0 |  |  |
| mis:if_statement->ProgramLooksAhead | not applied |  |  | 0.0 |  |  |

#### if_statement, level ways: 19 edges offered
Edges offered:
| edge |
|---|
| lvl:if_statement@L0#0 \| level L0 way everyday_reading: Folk meaning: English 'if' (and 'whenever') states a standing rule that stays in force and fires whenever its condition becomes true. A Java if is evaluated once, when control reaches it, and then is 'taken off the books'; changing the variables later does not re-trigger it. |
| lvl:if_statement@L0#14 \| level L0 way inconsistent_trace: Applies the if rule (LT) unevenly: tests the condition on some visits but not others (e.g. runs the branch on every loop pass without testing), runs the branch after a false test, or re-checks the condition after the branch has run. (here mixing in: "an if body repeats while its condition holds") |
| lvl:if_statement@L1#0 \| level L1 way names_the_element: The if statement is a conditional (selection) statement: it runs a statement or block only when a boolean condition is true. |
| lvl:if_statement@L1#1 \| level L1 way states_the_rule: Syntax: if (<boolean expression>) <statement or { block }>; the braces are optional when the body is a single statement. The expression must have type boolean (or Boolean), and the body is executed if and only if the expression is true. |
| lvl:if_statement@L2#0 \| level L2 way line_by_line_trace: Evaluate the condition once; if true, execute the body; then, in either case, continue with the statement after the if. |
| lvl:if_statement@L2#1 \| level L2 way output_prediction: An if produces no value; its observable result is that the body's effects (assignments, output) happen exactly once when the condition is true and not at all when it is false; e.g. if (x < 0) x = -x; leaves x = 5 for 5 and gives 5 for -5. |
| lvl:if_statement@L2#2 \| level L2 way one_step_execution: One execution reads the variables in the condition, changes nothing by the test itself (unless the condition has a side effect), and, when true, applies the body's changes once; control then moves to the next statement. |
| lvl:if_statement@L3#0 \| level L3 way relation_to_neighbour: The condition must be a boolean expression; a non-boolean condition such as if (t = goal) with int t is a compile-time error, while if (b = false) with boolean b compiles and assigns. |
| lvl:if_statement@L3#1 \| level L3 way relation_to_neighbour: if-else extends if with a second branch that runs exactly when the condition is false; a lone if has an implicit empty else-case. |
| lvl:if_statement@L3#2 \| level L3 way relation_to_neighbour: An if is a fork in the control flow: one path through the body, one around it, both rejoining at the next statement. |
| lvl:if_statement@L3#3 \| level L3 way relation_to_neighbour: if and while have the same form, but if tests its condition once and runs the block at most once, while while tests again after every pass and repeats the block as long as the condition is true. |
| lvl:if_statement@L3#4 \| level L3 way purpose_of_the_whole: Plan: guard / conditional action - do something only for the inputs that need it (absolute value, put two values in order, error check before dividing). |
| lvl:if_statement@L4#0 \| level L4 way predict_a_change: Negating the condition swaps which inputs run the body; changing < to <= changes only the boundary value. Adding a statement under a braceless if does not put it in the body. Whatever the condition, code after the if still runs exactly once. |
| lvl:if_statement@L4#1 \| level L4 way diagnose_an_error: rule broken: A ; right after if (...) is an empty statement that forms the whole then branch; the following statement or block is no longer part of the if. Kind: silent logic error. Trigger class: Any if (cond); followed by a statement or block: that statement runs for every input, whatever cond is (it compiles without error). |
| lvl:if_statement@L4#2 \| level L4 way diagnose_an_error: rule broken: The if template is if (<boolean expression>) <statement>: the condition must be enclosed in parentheses. Kind: compile-time. Trigger class: Any if whose condition is written in { } or without ( ), e.g. if {x > 0} ... or if a > b { c = 0; }. |
| lvl:if_statement@L4#3 \| level L4 way transfer_to_a_new_program: Use an if when an action is needed for some inputs only and the test happens once at that point; use a loop if the action must repeat while the condition holds, and assign a boolean directly (ok = x > y;) when you only need the truth value. |
| lvl:if_statement@L4#4 \| level L4 way diagnose_an_error: rule broken: Without braces, the body of an if, while or for is exactly one statement; indentation does not add statements to it. Kind: silent logic error. Trigger class: A braceless header followed by two or more indented statements: only the first is controlled, the others run once, after the if/loop, on every input. |
| lvl:if_statement@L4#5 \| level L4 way diagnose_an_error: rule broken: The condition of an if must have type boolean. Kind: compile-time. Trigger class: Using = instead of == with int operands, e.g. if (t = goal): 'incompatible types: int cannot be converted to boolean'. |
| lvl:if_statement@L4#6 \| level L4 way diagnose_an_error: rule broken: With boolean operands, = is assignment, not comparison. Kind: silent logic error. Trigger class: if (isPrime = false) with boolean isPrime: compiles, sets isPrime to false and the branch never runs. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:if_statement@L2#1 (Score: 0.4)}
{lvl:if_statement@L3#1 (Score: 0.35)}
{lvl:if_statement@L0#14 (Score: 0.25)}
```
Kept by pruning: lvl:if_statement@L2#1 (0.40), lvl:if_statement@L3#1 (0.35), lvl:if_statement@L0#14 (0.25)

Label prompt for level way lvl:if_statement@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> An if produces no value; its observable result is that the body's effects (assignments, output) happen exactly once when the condition is true and not at all when it is false; e.g. if (x < 0) x = -x; leaves x = 5 for 5 and gives 5 for -5.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:if_statement@L3#1 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> if-else extends if with a second branch that runs exactly when the condition is false; a lone if has an implicit empty else-case.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:if_statement@L0#14 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the if rule (LT) unevenly: tests the condition on some visits but not others (e.g. runs the branch on every loop pass without testing), runs the branch after a false test, or re-checks the condition after the branch has run. (here mixing in: "an if body repeats while its condition holds")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:if_statement@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:if_statement@L3#1 | not applied |  |  | 0.0 |  |  |
| lvl:if_statement@L0#14 | not applied |  |  | 0.0 |  |  |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

lvl:double@L3#0 [shown, tier 2, 0.35], lvl:arithmetic_operators@L4#0 [shown, tier 2, 0.34], lvl:double@L1#1 [shown, tier 2, 0.21]

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | not covered yet |
| c2 | The cast makes it a double division | not covered yet |
| c3 | so you get the decimal part | not covered yet |

**Uncovered clauses bring in (BM25 of the uncovered clauses against unvisited concepts):** println (5.41), boolean_expression (4.02), floating_point_precision (3.16)

**Graph check, not yet read (the walk may not stop while these remain):** own edges of boolean_expression, own edges of floating_point_precision, own edges of println, pre:arithmetic_operators->expression, pre:boolean_expression->if_statement, pre:control_flow->if_statement, pre:double->floating_point_precision, pre:if_statement->if_else_statement

**Figure 4 (the graph after depth 1)**

![Figure 4](figure04.png)
### Depth 2: current concepts 'double', 'arithmetic_operators', 'if_statement', 'println', 'boolean_expression', 'floating_point_precision'

#### double, concept edges: 1 edges offered
Edges offered:
| edge |
|---|
| pre:double->floating_point_precision \| rule: A double approximates real numbers, so its arithmetic is not exact. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:double->floating_point_precision (Score: 1.0)}
```
Kept by pruning: pre:double->floating_point_precision (1.00)

Label prompt for concept edge pre:double->floating_point_precision (double -> floating_point_precision, prerequisite):
> A double approximates real numbers, so its arithmetic is not exact.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.75
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:double->floating_point_precision | not applied |  |  | 0.0 |  |  |

#### arithmetic_operators, concept edges: 1 edges offered
Edges offered:
| edge |
|---|
| pre:arithmetic_operators->expression \| rule: An operator combines operand values a and b into a new value, so operators build larger expressions from smaller ones. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:arithmetic_operators->expression (Score: 1.0)}
```
Kept by pruning: pre:arithmetic_operators->expression (1.00)

Label prompt for concept edge pre:arithmetic_operators->expression (arithmetic_operators -> expression, prerequisite):
> An operator combines operand values a and b into a new value, so operators build larger expressions from smaller ones.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.55
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:arithmetic_operators->expression | not applied |  |  | 0.0 |  |  |

#### if_statement, concept edges: 3 edges offered
Edges offered:
| edge |
|---|
| pre:boolean_expression->if_statement \| rule: An if runs its block only when its boolean expression evaluates to true. |
| pre:control_flow->if_statement \| rule: An if changes the default top-to-bottom flow by running a block only when its condition is true. |
| pre:if_statement->if_else_statement \| rule: if-else extends if with a second block that runs exactly when the condition is false. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:if_statement->if_else_statement (Score: 0.5)}
{pre:boolean_expression->if_statement (Score: 0.3)}
{pre:control_flow->if_statement (Score: 0.2)}
```
Kept by pruning: pre:if_statement->if_else_statement (0.50), pre:boolean_expression->if_statement (0.30), pre:control_flow->if_statement (0.20)

Label prompt for concept edge pre:if_statement->if_else_statement (if_statement -> if_else_statement, prerequisite):
> if-else extends if with a second block that runs exactly when the condition is false.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.8
```

Label prompt for concept edge pre:boolean_expression->if_statement (boolean_expression -> if_statement, prerequisite):
> An if runs its block only when its boolean expression evaluates to true.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.8
```

Label prompt for concept edge pre:control_flow->if_statement (control_flow -> if_statement, prerequisite):
> An if changes the default top-to-bottom flow by running a block only when its condition is true.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:if_statement->if_else_statement | not applied |  |  | 0.0 |  |  |
| pre:boolean_expression->if_statement | not applied |  |  | 0.0 |  |  |
| pre:control_flow->if_statement | not applied |  |  | 0.0 |  |  |

#### println, misconceptions: 6 edges offered
Edges offered:
| edge |
|---|
| err:println->PrintWithoutFinalNewline \| wrong rule: Ends the output with print and expects it to appear right away. \| right rule: Finish a line of print calls with println; some systems hold print output until a newline is printed. |
| mis:println->IdentifierCaseInsensitive \| wrong rule: System.out.Println and System.out.println name the same method. \| right rule: Java is case-sensitive: names of variables, methods and classes that differ only in capitalization are different names. |
| mis:println->ParenthesesOnlyIfArgument \| wrong rule: println with nothing to print can be written System.out.println; without the parentheses. \| right rule: A method call always needs parentheses, so an empty line is printed with System.out.println();. |
| mis:println->PrintEqualsPrintln \| wrong rule: print and println do the same thing, so print(s1); print(s2); puts s1 and s2 on separate lines. \| right rule: println ends the line after printing and print does not, so print(s1); print(s2); puts s1 and s2 on the same line. |
| mis:println->PrintNewLineFirst \| wrong rule: println(s) prints a newline and then s. \| right rule: println(s) prints s and then the newline. |
| mis:println->PrintedTextIsExecuted \| wrong rule: System.out.println("x is 5"); sets x to 5 (text inside the quotes is run as code). \| right rule: Characters inside double quotes are printed literally; println only displays them and changes no variable. Only an assignment such as x = 5; sets x. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:println->PrintedTextIsExecuted (0.60), mis:println->PrintNewLineFirst (0.20), mis:println->PrintEqualsPrintln (0.20)
```
Kept by pruning: mis:println->PrintedTextIsExecuted (0.60), mis:println->PrintNewLineFirst (0.20), mis:println->PrintEqualsPrintln (0.20)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:println->PrintedTextIsExecuted \| wrong rule: System.out.println("x is 5"); sets x to 5 (text inside the quotes is run as code). \| right rule: Characters inside double quotes are printed literally; println only displays them and changes no variable. Only an assignment such as x = 5; sets x. |
| mis:println->PrintNewLineFirst \| wrong rule: println(s) prints a newline and then s. \| right rule: println(s) prints s and then the newline. |
| mis:println->PrintEqualsPrintln \| wrong rule: print and println do the same thing, so print(s1); print(s2); puts s1 and s2 on separate lines. \| right rule: println ends the line after printing and print does not, so print(s1); print(s2); puts s1 and s2 on the same line. |
Model's reply:
```
mis:println->PrintedTextIsExecuted | not applied | clauses: none | confidence: 0.9
mis:println->PrintNewLineFirst | not applied | clauses: none | confidence: 0.9
mis:println->PrintEqualsPrintln | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:println->PrintedTextIsExecuted | not applied |  |  | 0.0 |  |  |
| mis:println->PrintNewLineFirst | not applied |  |  | 0.0 |  |  |
| mis:println->PrintEqualsPrintln | not applied |  |  | 0.0 |  |  |

#### println, level ways: 17 edges offered
Edges offered:
| edge |
|---|
| lvl:println@L0#0 \| level L0 way everyday_reading: Everyday reading: 'println' is just 'print', and 'new line' is something you do before writing. Java rule: println(x) prints x and then ends the line; println() alone just ends the line. |
| lvl:println@L0#3 \| level L0 way inconsistent_trace: Applies the println rule (print, then end line) unevenly: puts the newline before the text on one call and after it on another, or forgets that the next output starts on a new line. (here mixing in: "print and println behave the same (no newline difference)") |
| lvl:println@L1#0 \| level L1 way names_the_element: System.out.println is a library method that prints its argument followed by a line terminator ('print line'). |
| lvl:println@L1#1 \| level L1 way states_the_rule: println(String x) behaves as though it calls print(x) and then println(); println() terminates the current line by writing the line separator string. |
| lvl:println@L2#0 \| level L2 way line_by_line_trace: Evaluate the argument, print its characters at the current position, then write the line separator so the cursor moves to the beginning of the next line. |
| lvl:println@L2#1 \| level L2 way output_prediction: Output is the value's characters followed by a line break: println("Hello, "); println("World!"); prints two lines. |
| lvl:println@L2#2 \| level L2 way one_step_execution: One call appends the text and a line separator to the output; the cursor ends at the start of the next line; no variable changes and nothing is returned. |
| lvl:println@L3#0 \| level L3 way relation_to_neighbour: System.out.println(s); is a statement, so its line of output appears at its own place in the sequence. |
| lvl:println@L3#1 \| level L3 way relation_to_neighbour: println("s") outputs exactly the characters between the quotes and then ends the line. |
| lvl:println@L3#2 \| level L3 way relation_to_neighbour: print leaves the cursor on the line; println ends it; several prints followed by one println make one line. |
| lvl:println@L3#3 \| level L3 way relation_to_neighbour: println prints numbers with all their digits and always ends the line; printf formats each value by its conversion code and ends the line only where the format has %n or \n. |
| lvl:println@L3#4 \| level L3 way purpose_of_the_whole: Purpose: produce one complete line of output, e.g. each result on its own line, or end a line built with print. |
| lvl:println@L4#0 \| level L4 way predict_a_change: Replacing println with print joins the next output onto the same line (a typical logic error); the characters printed stay the same. |
| lvl:println@L4#1 \| level L4 way transfer_to_a_new_program: Use println when each output item belongs on its own line or to end a line; use print to continue a line and printf to control number formatting. |
| lvl:println@L4#2 \| level L4 way diagnose_an_error: rule broken: Finish a line of print calls with println. Kind: silent logic error. Trigger class: output ends with print, so the last line may not be shown or runs into the shell prompt |
| lvl:println@L4#3 \| level L4 way diagnose_an_error: rule broken: Method names are case-sensitive. Kind: compile-time. Trigger class: System.out.Println or system.out.println: cannot find symbol |
| lvl:println@L4#4 \| level L4 way diagnose_an_error: rule broken: Calls need parentheses. Kind: compile-time. Trigger class: System.out.println; without () |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:println@L2#1 (Score: 0.4)}
{lvl:println@L3#1 (Score: 0.35)}
{lvl:println@L3#3 (Score: 0.25)}
```
Kept by pruning: lvl:println@L2#1 (0.40), lvl:println@L3#1 (0.35), lvl:println@L3#3 (0.25)

Label prompt for level way lvl:println@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> Output is the value's characters followed by a line break: println("Hello, "); println("World!"); prints two lines.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.6
```

Label prompt for level way lvl:println@L3#1 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> println("s") outputs exactly the characters between the quotes and then ends the line.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:println@L3#3 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> println prints numbers with all their digits and always ends the line; printf formats each value by its conversion code and ends the line only where the format has %n or \n.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:println@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:println@L3#1 | not applied |  |  | 0.0 |  |  |
| lvl:println@L3#3 | not applied |  |  | 0.0 |  |  |

#### boolean_expression, misconceptions: 8 edges offered
Edges offered:
| edge |
|---|
| err:boolean_expression->ElseWithCondition \| wrong rule: Writes else (x > 0) with a condition but no if. \| right rule: else takes no condition; use else if (x > 0). |
| mis:boolean_expression->ArithmeticInsteadOfBoolean \| wrong rule: A true/false question is answered with arithmetic (e.g. 1 for true). \| right rule: A logic question is written as a boolean expression whose value is true or false. |
| mis:boolean_expression->ChainedComparison \| wrong rule: 90 <= grade <= 100 tests whether grade is between 90 and 100. \| right rule: a < b < c does not compile (a < b is a boolean); a range test is written 90 <= grade && grade <= 100. |
| mis:boolean_expression->ComparisonWithBooleanLiteral \| wrong rule: To test a boolean b you must write if (b == true). \| right rule: if (b) already tests b, so writing == true adds nothing. |
| mis:boolean_expression->LoopForOneTimeTest \| wrong rule: Uses a loop where a condition needs to be tested only once. \| right rule: A test made once is an if; a loop repeats. |
| mis:boolean_expression->MapToBooleanWithConditionalOperator \| wrong rule: To get a boolean from x > y you need ok = (x > y) ? true : false;. \| right rule: ok = (x > y); already gives the boolean, so the ? : is unnecessary. |
| mis:boolean_expression->MapToBooleanWithIf \| wrong rule: To store whether x > y in a boolean you need if (x > y) ok = true; else ok = false;. \| right rule: ok = (x > y); stores the result of the comparison directly. |
| mis:boolean_expression->NestedIfsInsteadOfLogicalOperators \| wrong rule: A condition on several values has to be written as nested if statements, one value per if. \| right rule: Several conditions combine into one boolean expression with && and \|\|, e.g. if (x == 0 && y == 0). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:boolean_expression->ArithmeticInsteadOfBoolean (0.80), mis:boolean_expression->MapToBooleanWithIf (0.10), mis:boolean_expression->ComparisonWithBooleanLiteral (0.10)
```
Kept by pruning: mis:boolean_expression->ArithmeticInsteadOfBoolean (0.80), mis:boolean_expression->MapToBooleanWithIf (0.10), mis:boolean_expression->ComparisonWithBooleanLiteral (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:boolean_expression->ArithmeticInsteadOfBoolean \| wrong rule: A true/false question is answered with arithmetic (e.g. 1 for true). \| right rule: A logic question is written as a boolean expression whose value is true or false. |
| mis:boolean_expression->MapToBooleanWithIf \| wrong rule: To store whether x > y in a boolean you need if (x > y) ok = true; else ok = false;. \| right rule: ok = (x > y); stores the result of the comparison directly. |
| mis:boolean_expression->ComparisonWithBooleanLiteral \| wrong rule: To test a boolean b you must write if (b == true). \| right rule: if (b) already tests b, so writing == true adds nothing. |
Model's reply:
```
mis:boolean_expression->ArithmeticInsteadOfBoolean | not applied | clauses: none | confidence: 0.85
mis:boolean_expression->MapToBooleanWithIf | not applied | clauses: none | confidence: 0.95
mis:boolean_expression->ComparisonWithBooleanLiteral | not applied | clauses: none | confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:boolean_expression->ArithmeticInsteadOfBoolean | not applied |  |  | 0.0 |  |  |
| mis:boolean_expression->MapToBooleanWithIf | not applied |  |  | 0.0 |  |  |
| mis:boolean_expression->ComparisonWithBooleanLiteral | not applied |  |  | 0.0 |  |  |

#### boolean_expression, level ways: 19 edges offered
Edges offered:
| edge |
|---|
| lvl:boolean_expression@L0#0 \| level L0 way everyday_reading: Folk meaning: maths writes a range as 90 <= grade <= 100 and English says 'if x is 1 or 2'. Java rule: each comparison is a separate boolean expression and they must be joined explicitly: 90 <= grade && grade <= 100, x == 1 \|\| x == 2; 90 <= grade <= 100 does not compile because 90 <= grade is already a boolean. |
| lvl:boolean_expression@L0#5 \| level L0 way inconsistent_trace: Points to LT (arithmetic first, then comparisons, then !, &&, \|\|, left to right, with short-circuit). Applied unevenly: the student evaluates && before a comparison in one sub-expression, or evaluates the right operand of && on a pass where the left was already false. (here mixing in: "you must write == true / == false to test a boolean") |
| lvl:boolean_expression@L1#0 \| level L1 way names_the_element: A boolean expression is any expression whose type is boolean: a boolean literal or variable, a comparison, or such parts combined with &&, \|\| and !. |
| lvl:boolean_expression@L1#1 \| level L1 way states_the_rule: A boolean expression has value true or false; it is built from comparisons (lower precedence than arithmetic, higher than boolean operators) and boolean operators; the condition of if, while and for must be one. |
| lvl:boolean_expression@L2#0 \| level L2 way line_by_line_trace: Operands are evaluated left to right; arithmetic sub-expressions first, then comparisons, then !, then &&, then \|\|; && and \|\| evaluate their right operand only when the left has not decided the result. |
| lvl:boolean_expression@L2#1 \| level L2 way output_prediction: The value is true or false: with year = 1900, (year % 4 == 0) && (year % 100 != 0) \|\| (year % 400 == 0) is false, so LeapYear prints false. |
| lvl:boolean_expression@L2#2 \| level L2 way one_step_execution: Evaluating a boolean expression changes no variable (unless it contains an assignment, ++ or a method with side effects); state changes only when its value is stored, as in isLeapYear = isLeapYear && (year % 100 != 0);. |
| lvl:boolean_expression@L3#0 \| level L3 way relation_to_neighbour: Comparisons are the atoms of boolean expressions; because they bind tighter than && and \|\|, month >= 1 && month <= 12 groups as (month >= 1) && (month <= 12) without parentheses. |
| lvl:boolean_expression@L3#1 \| level L3 way relation_to_neighbour: &&, \|\| and ! combine boolean sub-expressions into a larger one; ! binds tighter than comparisons, so negating a comparison needs parentheses: !(x > 0). |
| lvl:boolean_expression@L3#2 \| level L3 way relation_to_neighbour: The if statement evaluates its boolean expression once and runs its block only when the value is true; the expression must have type boolean, so if (x = 5) with int x does not compile. |
| lvl:boolean_expression@L3#3 \| level L3 way relation_to_neighbour: The while loop re-evaluates its boolean expression before every pass and stops the first time it is false, so some variable in the expression must change in the body or the loop never ends. |
| lvl:boolean_expression@L3#4 \| level L3 way relation_to_neighbour: In RandomSurfer the boolean expression r < sum is evaluated for each candidate page j in turn, as sum grows; the first j for which it is true is the chosen page. |
| lvl:boolean_expression@L3#5 \| level L3 way relation_to_neighbour: In the template if (<boolean expression>) { <statements> } the placeholder may be filled by any boolean-typed expression: a literal, a boolean variable, a comparison or a combination with logical operators. |
| lvl:boolean_expression@L3#6 \| level L3 way purpose_of_the_whole: Expresses a decision condition or guard (which branch runs, whether a loop continues) or computes a flag; common plans: range test lo <= x && x <= hi, divisibility test n % k == 0, combined rule as in LeapYear. |
| lvl:boolean_expression@L4#0 \| level L4 way predict_a_change: Rewriting with De Morgan (!(a && b) as !a \|\| !b) keeps the value for every input; swapping the operands of && keeps the truth value but can change whether the right operand is evaluated (and whether a guard still protects it); changing < to <= changes the result only at the boundary value. |
| lvl:boolean_expression@L4#1 \| level L4 way diagnose_an_error: rule broken: else takes no condition; a second condition needs else if (...). Kind: compile-time. Trigger class: Writing else (x > 0) { ... } after an if block (the compiler reports e.g. 'not a statement' or '; expected'). |
| lvl:boolean_expression@L4#2 \| level L4 way transfer_to_a_new_program: Use a boolean expression wherever code must decide or record a yes/no condition, and store it directly (ok = x > y) rather than via if or ? :; when different outcomes are needed for several cases use an if-else chain of such expressions. |
| lvl:boolean_expression@L4#3 \| level L4 way diagnose_an_error: rule broken: The condition must be a boolean expression; = assigns, == compares. Kind: compile-time. Trigger class: if (a = b) with int a and b (with boolean variables it compiles and silently assigns). |
| lvl:boolean_expression@L4#4 \| level L4 way diagnose_an_error: rule broken: Comparisons cannot be chained; each must be joined with && or \|\|. Kind: compile-time. Trigger class: Range written as a < b < c or 90 <= grade <= 100. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:boolean_expression@L2#1 (Score: 0.5)}
{lvl:boolean_expression@L3#2 (Score: 0.3)}
{lvl:boolean_expression@L0#5 (Score: 0.2)}
```
Kept by pruning: lvl:boolean_expression@L2#1 (0.50), lvl:boolean_expression@L3#2 (0.30), lvl:boolean_expression@L0#5 (0.20)

Label prompt for level way lvl:boolean_expression@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The value is true or false: with year = 1900, (year % 4 == 0) && (year % 100 != 0) || (year % 400 == 0) is false, so LeapYear prints false.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:boolean_expression@L3#2 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> The if statement evaluates its boolean expression once and runs its block only when the value is true; the expression must have type boolean, so if (x = 5) with int x does not compile.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:boolean_expression@L0#5 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Points to LT (arithmetic first, then comparisons, then !, &&, ||, left to right, with short-circuit). Applied unevenly: the student evaluates && before a comparison in one sub-expression, or evaluates the right operand of && on a pass where the left was already false. (here mixing in: "you must write == true / == false to test a boolean")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:boolean_expression@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:boolean_expression@L3#2 | not applied |  |  | 0.0 |  |  |
| lvl:boolean_expression@L0#5 | not applied |  |  | 0.0 |  |  |

#### floating_point_precision, misconceptions: 1 edges offered
Edges offered:
| edge |
|---|
| mis:floating_point_precision->FloatingPointIsExact \| wrong rule: Math.sqrt(2) * Math.sqrt(2) == 2 is true. \| right rule: A double is an approximation, so the product is close to 2 but not equal to 2. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{mis:floating_point_precision->FloatingPointIsExact (Score: 1.0)}
```
Kept by pruning: mis:floating_point_precision->FloatingPointIsExact (1.00)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:floating_point_precision->FloatingPointIsExact \| wrong rule: Math.sqrt(2) * Math.sqrt(2) == 2 is true. \| right rule: A double is an approximation, so the product is close to 2 but not equal to 2. |
Model's reply:
```
mis:floating_point_precision->FloatingPointIsExact | not applied | clauses: none | confidence: 0.85
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:floating_point_precision->FloatingPointIsExact | not applied |  |  | 0.0 |  |  |

#### floating_point_precision, level ways: 14 edges offered
Edges offered:
| edge |
|---|
| lvl:floating_point_precision@L0#0 \| level L0 way everyday_reading: Folk meaning: decimal arithmetic is exact, as on paper or on a calculator display, so 0.1 + 0.1 + 0.1 equals 0.3. Java rule: most decimal fractions (0.1) have no exact binary representation, every double operation rounds, and (0.1 + 0.1 + 0.1 == 0.3) is false. |
| lvl:floating_point_precision@L0#2 \| level L0 way inconsistent_trace: Points to LT (rounding happens after every single operation, and the error is carried into the next one). Applied unevenly: the student rounds only the final printed value, treating intermediate values as exact on some lines, or expects ten additions of 0.1 to equal 0.1 * 10. (here mixing in: "Math.sqrt(2) * Math.sqrt(2) == 2 is true") |
| lvl:floating_point_precision@L1#0 \| level L1 way names_the_element: Floating-point precision is a property of the double type: a double keeps only about 15 significant decimal digits, and the difference between the intended real number and the stored double is rounding error. |
| lvl:floating_point_precision@L1#1 \| level L1 way states_the_rule: Each double result is the nearest representable binary64 value, so results are generally approximate; S&W: novice programmers should avoid comparing two floating-point numbers for equality. Do not test doubles for equality with ==: compare with a tolerance, e.g. Math.abs(x - y) < 1e-9; and == is false whenever an operand is NaN. |
| lvl:floating_point_precision@L2#0 \| level L2 way line_by_line_trace: Evaluation proceeds operator by operator; after each one the exact result is rounded to a double, and the next operator works on that rounded value, so errors compound along the evaluation order; only at output is the stored value converted to a decimal string. |
| lvl:floating_point_precision@L2#1 \| level L2 way output_prediction: Printed results show the approximation: 0.1 * 10 prints 1.0 but adding 0.1 ten times prints 0.9999999999999999; 5.0/3.0 prints 1.6666666666666667; (0.1 + 0.1 + 0.1 == 0.3) is false. |
| lvl:floating_point_precision@L2#2 \| level L2 way one_step_execution: Each assignment such as x = x + 0.1 stores the rounded sum; the error stays in x and is carried into every later step that uses x. |
| lvl:floating_point_precision@L3#0 \| level L3 way relation_to_neighbour: Imprecision is inherent in the double type, not in a particular program: any arithmetic on doubles may carry rounding error, so code that uses double must be written to tolerate it. |
| lvl:floating_point_precision@L3#1 \| level L3 way relation_to_neighbour: Because doubles are approximate, a loop that waits for exact equality may never stop; Sqrt therefore loops while Math.abs(t - c/t) > epsilon * t, a relative-tolerance test. |
| lvl:floating_point_precision@L3#2 \| level L3 way relation_to_neighbour: Both come from finite representations: an int result too large for 32 bits wraps around silently (overflow), while a double keeps about 15 significant digits so most results are approximations (5.0/3.0 is 1.6666666666666667). Neither raises an error. |
| lvl:floating_point_precision@L3#3 \| level L3 way purpose_of_the_whole: Awareness of precision is used to write tolerance comparisons (Math.abs(a - b) < epsilon) and to choose integer representations for exact quantities, e.g. money as an int number of cents. |
| lvl:floating_point_precision@L4#0 \| level L4 way predict_a_change: Reordering additions, or changing a loop step such as x += 0.1, changes the last digits and can change how many times a loop tested with <= runs; replacing an == test by a tolerance test makes the outcome stable under such changes. |
| lvl:floating_point_precision@L4#1 \| level L4 way transfer_to_a_new_program: Accept double precision for measurements and simulations and compare with a tolerance; when exactness is required (counts, money, loop control) use int or long instead. |
| lvl:floating_point_precision@L4#2 \| level L4 way diagnose_an_error: rule broken: Computed doubles must not be compared for exact equality. Kind: silent logic error. Trigger class: == or != between doubles from arithmetic on non-dyadic fractions (0.1, 1/3, square roots), or a double loop counter with a == / <= bound; can also cause an infinite loop. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:floating_point_precision@L0#0 (Score: 0.5)}
{lvl:floating_point_precision@L2#1 (Score: 0.3)}
{lvl:floating_point_precision@L0#2 (Score: 0.2)}
```
Kept by pruning: lvl:floating_point_precision@L0#0 (0.50), lvl:floating_point_precision@L2#1 (0.30), lvl:floating_point_precision@L0#2 (0.20)

Label prompt for level way lvl:floating_point_precision@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: decimal arithmetic is exact, as on paper or on a calculator display, so 0.1 + 0.1 + 0.1 equals 0.3. Java rule: most decimal fractions (0.1) have no exact binary representation, every double operation rounds, and (0.1 + 0.1 + 0.1 == 0.3) is false.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:floating_point_precision@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> Printed results show the approximation: 0.1 * 10 prints 1.0 but adding 0.1 ten times prints 0.9999999999999999; 5.0/3.0 prints 1.6666666666666667; (0.1 + 0.1 + 0.1 == 0.3) is false.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:floating_point_precision@L0#2 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Points to LT (rounding happens after every single operation, and the error is carried into the next one). Applied unevenly: the student rounds only the final printed value, treating intermediate values as exact on some lines, or expects ten additions of 0.1 to equal 0.1 * 10. (here mixing in: "Math.sqrt(2) * Math.sqrt(2) == 2 is true")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.93
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:floating_point_precision@L0#0 | not applied |  |  | 0.0 |  |  |
| lvl:floating_point_precision@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:floating_point_precision@L0#2 | not applied |  |  | 0.0 |  |  |

**Kept on a path at this depth:** nothing

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | not covered yet |
| c2 | The cast makes it a double division | not covered yet |
| c3 | so you get the decimal part | not covered yet |

**Uncovered clauses bring in (BM25 of the uncovered clauses against unvisited concepts):** comparison_operators (2.85), control_flow (2.80), if_else_statement (1.78)

**Graph check, not yet read (the walk may not stop while these remain):** own edges of comparison_operators, own edges of control_flow, own edges of if_else_statement, pre:comparison_operators->boolean_expression, pre:statement->println, pre:string_literal->println

**Figure 5 (the graph after depth 2)**

![Figure 5](figure05.png)
### Depth 3: current concepts 'println', 'boolean_expression', 'comparison_operators', 'control_flow', 'if_else_statement'

#### println, concept edges: 2 edges offered
Edges offered:
| edge |
|---|
| pre:statement->println \| rule: System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence. |
| pre:string_literal->println \| rule: println("s") outputs exactly the characters between the quotes and then ends the line. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:string_literal->println (Score: 0.7)}
{pre:statement->println (Score: 0.3)}
```
Kept by pruning: pre:string_literal->println (0.70), pre:statement->println (0.30)

Label prompt for concept edge pre:string_literal->println (string_literal -> println, prerequisite):
> println("s") outputs exactly the characters between the quotes and then ends the line.

Model's reply:
```
Label: broken
Clauses: 1
Confidence: 0.55
```

Label prompt for concept edge pre:statement->println (statement -> println, prerequisite):
> System.out.println(s); is itself a statement, so its line of output appears at its own place in the sequence.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.8
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:string_literal->println | broken | 0.7 | 0.55 | 0.39 | c1 | 3.5 |
| pre:statement->println | not applied |  |  | 0.0 |  |  |

#### boolean_expression, concept edges: 2 edges offered
Edges offered:
| edge |
|---|
| pre:boolean_expression->if_statement \| rule: An if runs its block only when its boolean expression evaluates to true. |
| pre:comparison_operators->boolean_expression \| rule: A comparison such as x < y produces a boolean result; comparisons together with boolean logic provide the basis for decision-making. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{pre:boolean_expression->if_statement (Score: 0.6)}
{pre:comparison_operators->boolean_expression (Score: 0.4)}
```
Kept by pruning: pre:boolean_expression->if_statement (0.60), pre:comparison_operators->boolean_expression (0.40)

Label prompt for concept edge pre:boolean_expression->if_statement (boolean_expression -> if_statement, prerequisite):
> An if runs its block only when its boolean expression evaluates to true.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.8
```

Label prompt for concept edge pre:comparison_operators->boolean_expression (comparison_operators -> boolean_expression, prerequisite):
> A comparison such as x < y produces a boolean result; comparisons together with boolean logic provide the basis for decision-making.

Model's reply:
```
Label: not applied
Clauses: none
Confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| pre:boolean_expression->if_statement | not applied |  |  | 0.0 |  |  |
| pre:comparison_operators->boolean_expression | not applied |  |  | 0.0 |  |  |

#### comparison_operators, misconceptions: 8 edges offered
Edges offered:
| edge |
|---|
| err:comparison_operators->ReversedRelationalOperator \| wrong rule: Writes =< or => instead of <= and >=. \| right rule: The operators are written <= and >=. |
| mis:comparison_operators->AssignCompares \| wrong rule: = compares two values instead of assigning \| right rule: x = y copies the value of y into x; x == y is what compares them. |
| mis:comparison_operators->ChainedComparison \| wrong rule: 90 <= grade <= 100 tests whether grade is between 90 and 100. \| right rule: a < b < c does not compile (a < b is a boolean); a range test is written 90 <= grade && grade <= 100. |
| mis:comparison_operators->ComparisonWithBooleanLiteral \| wrong rule: you must write == true / == false to test a boolean \| right rule: if (b) already tests b, so writing == true adds nothing. |
| mis:comparison_operators->EqualityOperatorComparesObjectsValues \| wrong rule: s == t is true whenever the two Strings contain the same text. \| right rule: s == t is true only when s and t refer to the same object; s.equals(t) compares the text. |
| mis:comparison_operators->EqualsComparesReferences \| wrong rule: .equals() compares references, not contents \| right rule: s.equals(t) checks whether the two Strings contain the same characters. |
| mis:comparison_operators->LessGreaterSwapped \| wrong rule: < and > mean the same thing (or a < b means a is greater than b). \| right rule: a < b is true when a is smaller than b; a > b when a is larger. |
| mis:comparison_operators->StringRelationalComparison \| wrong rule: s < t compares two Strings alphabetically. \| right rule: < and > are defined only for primitive types; comparing Strings with them does not compile. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:comparison_operators->EqualityOperatorComparesObjectsValues (0.40), mis:comparison_operators->AssignCompares (0.30), mis:comparison_operators->EqualsComparesReferences (0.30)
```
Kept by pruning: mis:comparison_operators->EqualityOperatorComparesObjectsValues (0.40), mis:comparison_operators->AssignCompares (0.30), mis:comparison_operators->EqualsComparesReferences (0.30)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:comparison_operators->EqualityOperatorComparesObjectsValues \| wrong rule: s == t is true whenever the two Strings contain the same text. \| right rule: s == t is true only when s and t refer to the same object; s.equals(t) compares the text. |
| mis:comparison_operators->AssignCompares \| wrong rule: = compares two values instead of assigning \| right rule: x = y copies the value of y into x; x == y is what compares them. |
| mis:comparison_operators->EqualsComparesReferences \| wrong rule: .equals() compares references, not contents \| right rule: s.equals(t) checks whether the two Strings contain the same characters. |
Model's reply:
```
mis:comparison_operators->EqualityOperatorComparesObjectsValues | not applied | clauses: none | confidence: 0.95
mis:comparison_operators->AssignCompares | not applied | clauses: none | confidence: 0.95
mis:comparison_operators->EqualsComparesReferences | not applied | clauses: none | confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:comparison_operators->EqualityOperatorComparesObjectsValues | not applied |  |  | 0.0 |  |  |
| mis:comparison_operators->AssignCompares | not applied |  |  | 0.0 |  |  |
| mis:comparison_operators->EqualsComparesReferences | not applied |  |  | 0.0 |  |  |

#### comparison_operators, level ways: 20 edges offered
Edges offered:
| edge |
|---|
| lvl:comparison_operators@L0#0 \| level L0 way everyday_reading: Folk meaning: in maths = states that two things are equal, and inequalities can be chained (a < b < c) or spoken as 'equal or less than' (=<). Java rule: = assigns while == compares; the operators are spelled <= and >= only; a < b < c does not compile; and on Strings == compares references. |
| lvl:comparison_operators@L0#5 \| level L0 way inconsistent_trace: Points to LT (evaluate both operands, promote to a common type, then compare). Applied unevenly: the student treats < as <= on the boundary pass of a loop, or compares an int with a double as ints on one line and as doubles on another. (here mixing in: "90 <= grade <= 100 tests whether grade is between 90 and 100") |
| lvl:comparison_operators@L1#0 \| level L1 way names_the_element: Comparison (relational) operators ==, !=, <, <=, >, >= are mixed-type operators: they take two operands of a numeric type and produce a boolean. |
| lvl:comparison_operators@L1#1 \| level L1 way states_the_rule: ==, !=, <, <=, >, >= are defined for each primitive numeric type and produce a boolean; they have lower precedence than arithmetic and higher than boolean operators; < and > are not defined for Strings, and == on non-primitive types compares identity. |
| lvl:comparison_operators@L2#0 \| level L2 way line_by_line_trace: Evaluate the left operand (including its arithmetic), then the right operand; if their types differ, promote the narrower one (int to double); then compare and produce true or false. |
| lvl:comparison_operators@L2#1 \| level L2 way output_prediction: The result is a boolean: (2 <= 2) is true, 2 < 2 is false, (x < 3.1) is true for int x = 3; any comparison involving NaN, even NaN == NaN, is false. |
| lvl:comparison_operators@L2#2 \| level L2 way one_step_execution: A comparison changes no variable and only returns a boolean, unlike =, which changes the value of the variable on its left. |
| lvl:comparison_operators@L3#0 \| level L3 way relation_to_neighbour: A comparison is itself an expression whose value is a boolean, so it can appear anywhere a boolean is needed, including on the right of an assignment (boolean ok = x > y;). |
| lvl:comparison_operators@L3#1 \| level L3 way relation_to_neighbour: Comparisons are the building blocks of conditions: together with boolean logic they provide the basis for decision-making. |
| lvl:comparison_operators@L3#2 \| level L3 way relation_to_neighbour: Comparisons bind tighter than && and \|\|, so they are evaluated first and logical operators combine their boolean results. |
| lvl:comparison_operators@L3#3 \| level L3 way relation_to_neighbour: Comparisons take numbers and produce a boolean; boolean values themselves can be compared only with == and != (not <), and testing them against true is unnecessary. |
| lvl:comparison_operators@L3#4 \| level L3 way relation_to_neighbour: x = y stores the value of y in x (a statement with a side effect); x == y only tests whether the two values are equal and gives a boolean. Writing = where == is meant is a compile error with int operands and a silent bug with boolean operands. |
| lvl:comparison_operators@L3#5 \| level L3 way purpose_of_the_whole: Used to test a numeric condition: thresholds and sign checks, range tests, divisibility (n % k == 0), loop continuation (i < N), sentinel checks, and running-extreme updates (if (x > max) max = x). |
| lvl:comparison_operators@L4#0 \| level L4 way predict_a_change: Changing < to <= (or > to >=) changes the result only at the boundary value, which in a loop guard means one more or one fewer iteration (off-by-one); mirroring the comparison (a < b to b > a) preserves the value; negating it gives the complementary operator (!(a < b) is a >= b). |
| lvl:comparison_operators@L4#1 \| level L4 way diagnose_an_error: rule broken: The operators are written <= and >=. Kind: compile-time. Trigger class: Writing =< or => (e.g. if (x => 0)). |
| lvl:comparison_operators@L4#2 \| level L4 way transfer_to_a_new_program: Use comparison operators for numeric and char tests; for Strings use equals (or compareTo); for computed doubles use a tolerance; for booleans use the variable itself instead of == true. |
| lvl:comparison_operators@L4#3 \| level L4 way diagnose_an_error: rule broken: == compares, = assigns. Kind: compile-time. Trigger class: if (a = b) with int operands (with boolean operands it compiles and silently assigns). |
| lvl:comparison_operators@L4#4 \| level L4 way diagnose_an_error: rule broken: == on Strings compares references, not text. Kind: silent logic error. Trigger class: Comparing a String built at run time (input, concatenation) with a literal using ==. |
| lvl:comparison_operators@L4#5 \| level L4 way diagnose_an_error: rule broken: < and > are defined only for primitive types; comparisons cannot be chained; both operands must be compatible. Kind: compile-time. Trigger class: s < t on Strings, a < b < c, or 5 < "6". |
| lvl:comparison_operators@L4#6 \| level L4 way diagnose_an_error: rule broken: Computed doubles should not be compared with ==. Kind: silent logic error. Trigger class: == between doubles produced by arithmetic, e.g. (0.1 + 0.1 + 0.1 == 0.3). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:comparison_operators@L4#6 (Score: 0.4)}
{lvl:comparison_operators@L2#0 (Score: 0.35)}
{lvl:comparison_operators@L2#1 (Score: 0.25)}
```
Kept by pruning: lvl:comparison_operators@L4#6 (0.40), lvl:comparison_operators@L2#0 (0.35), lvl:comparison_operators@L2#1 (0.25)

Label prompt for level way lvl:comparison_operators@L4#6 (the model saw only the concept, the way's name 'diagnose an error' and its text; no level):
> rule broken: Computed doubles should not be compared with ==. Kind: silent logic error. Trigger class: == between doubles produced by arithmetic, e.g. (0.1 + 0.1 + 0.1 == 0.3).

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.97
```

Label prompt for level way lvl:comparison_operators@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> Evaluate the left operand (including its arithmetic), then the right operand; if their types differ, promote the narrower one (int to double); then compare and produce true or false.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.9
```

Label prompt for level way lvl:comparison_operators@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The result is a boolean: (2 <= 2) is true, 2 < 2 is false, (x < 3.1) is true for int x = 3; any comparison involving NaN, even NaN == NaN, is false.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.92
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:comparison_operators@L4#6 | not applied |  |  | 0.0 |  |  |
| lvl:comparison_operators@L2#0 | not applied |  |  | 0.0 |  |  |
| lvl:comparison_operators@L2#1 | not applied |  |  | 0.0 |  |  |

#### control_flow, misconceptions: 11 edges offered
Edges offered:
| edge |
|---|
| mis:control_flow->CodeAfterIfSkipped \| wrong rule: When the then branch runs, the code after the if is skipped. \| right rule: After the if finishes, execution continues with the next statement either way. |
| mis:control_flow->FalseConditionRestartsProgram \| wrong rule: When an if condition is false (and there is no else), control jumps back to the start of the program. \| right rule: A false condition skips the then branch and execution continues with the statement after the if statement. |
| mis:control_flow->FalseLoopRepeatsNextCode \| wrong rule: A loop is an iterative if-else: while the condition is true the body repeats, and when it is false the code after the loop repeats. \| right rule: Only the body is repeated; once the condition is false the code after the loop runs once. |
| mis:control_flow->IfConditionWatched \| wrong rule: An if statement watches its condition while the program runs and fires as soon as the condition becomes true (if = whenever). \| right rule: An if condition is tested once, when control reaches the if statement; later changes to the variables do not trigger the branch. |
| mis:control_flow->IfWaitsForCondition \| wrong rule: An IF statement earlier in the program stays active and fires later, e.g. inside a later loop, when its condition becomes true. \| right rule: An if is evaluated once when control reaches it; afterwards it is inactive unless control returns to it. |
| mis:control_flow->LoopImpliedByOneTest \| wrong rule: Testing one case is enough; Java will repeat the test for the other values by itself. \| right rule: A statement runs once each time control reaches it; repeating a test for many values needs a loop. |
| mis:control_flow->NoShortCircuit \| wrong rule: logical AND/OR always evaluate both sides (Java short-circuits them) \| right rule: In p && q, q is evaluated only when p is true; in p \|\| q, q is evaluated only when p is false. |
| mis:control_flow->ProgramEndsAfterLoop \| wrong rule: When a loop finishes the program ends, so code after the loop never runs. \| right rule: When the loop condition becomes false, control continues with the first statement after the loop. |
| mis:control_flow->ProgramLooksAhead \| wrong rule: The program knows or sees what later lines will do and acts on those goals at the current line, e.g. it stops 'at ten' because it wants to draw a box. \| right rule: Each statement executes using only the current state; later lines have no effect until control reaches them. |
| mis:control_flow->ReturnDoesNotExitLoop \| wrong rule: A return inside a loop does not end the loop; it sets a value each pass (the last one is returned) or returns several values. \| right rule: When return executes, the method ends at once, so the loop stops on that pass. |
| mis:control_flow->StatementsRunInAnyOrder \| wrong rule: Statements in a program can run in any order. \| right rule: Without control statements, statements run once each, top to bottom. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:control_flow->StatementsRunInAnyOrder (0.40), mis:control_flow->CodeAfterIfSkipped (0.30), mis:control_flow->FalseConditionRestartsProgram (0.30)
```
Kept by pruning: mis:control_flow->StatementsRunInAnyOrder (0.40), mis:control_flow->CodeAfterIfSkipped (0.30), mis:control_flow->FalseConditionRestartsProgram (0.30)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:control_flow->StatementsRunInAnyOrder \| wrong rule: Statements in a program can run in any order. \| right rule: Without control statements, statements run once each, top to bottom. |
| mis:control_flow->CodeAfterIfSkipped \| wrong rule: When the then branch runs, the code after the if is skipped. \| right rule: After the if finishes, execution continues with the next statement either way. |
| mis:control_flow->FalseConditionRestartsProgram \| wrong rule: When an if condition is false (and there is no else), control jumps back to the start of the program. \| right rule: A false condition skips the then branch and execution continues with the statement after the if statement. |
Model's reply:
```
mis:control_flow->StatementsRunInAnyOrder | not applied | clauses: none | confidence: 0.9
mis:control_flow->CodeAfterIfSkipped | not applied | clauses: none | confidence: 0.9
mis:control_flow->FalseConditionRestartsProgram | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:control_flow->StatementsRunInAnyOrder | not applied |  |  | 0.0 |  |  |
| mis:control_flow->CodeAfterIfSkipped | not applied |  |  | 0.0 |  |  |
| mis:control_flow->FalseConditionRestartsProgram | not applied |  |  | 0.0 |  |  |

#### control_flow, level ways: 15 edges offered
Edges offered:
| edge |
|---|
| lvl:control_flow@L0#0 \| level L0 way everyday_reading: Folk meaning: in everyday instructions a listener reads the whole list at once, does steps in a sensible order and infers what the writer wants. Java instead executes statements one at a time, top to bottom, each only when control reaches it, and nothing later in the program has any effect earlier. |
| lvl:control_flow@L0#3 \| level L0 way inconsistent_trace: Applies the sequencing rule (LT) unevenly: e.g. runs a later line before an earlier one, lets an if that was already passed fire again later, or ends the program when a loop or a false if ends instead of continuing with the next statement. (here mixing in: "Statements in a program can run in any order") |
| lvl:control_flow@L1#0 \| level L1 way names_the_element: Control flow is the order in which a program's statements are executed (statement sequencing); conditionals and loops are the statements that change it. |
| lvl:control_flow@L1#1 \| level L1 way states_the_rule: Without control statements every statement executes once, in the order given; a block executes its statements in order from first to last; if/if-else choose between one or two alternatives and loops repeat statements depending on a condition. |
| lvl:control_flow@L2#0 \| level L2 way line_by_line_trace: Execute the statement at the current point; when it completes normally, control passes to the next statement in the block. An if evaluates its condition once and then runs at most one branch; a loop evaluates its condition before each pass; after an if or a loop finishes, control continues with the statement after it. |
| lvl:control_flow@L2#1 \| level L2 way output_prediction: The output is the sequence of print statements that control actually reaches, in the order reached: e.g. TenHellos prints 1st, 2nd, 3rd Hello once each, then 4th to 10th Hello from the loop. |
| lvl:control_flow@L2#2 \| level L2 way one_step_execution: One step executes the single statement at the current control point using the current values, updates at most the variables it assigns, and moves the control point to the next statement (or, for an if/loop test, to the chosen branch, back to the test, or past the statement). |
| lvl:control_flow@L3#0 \| level L3 way relation_to_neighbour: Control flow is defined over statements: statements are executed for their effect, and if/while/for are themselves statements that may be used wherever a statement is allowed, so they can nest. |
| lvl:control_flow@L3#1 \| level L3 way relation_to_neighbour: An if changes the default top-to-bottom flow: its condition is tested once when control reaches it, its block runs only if true, and in both cases control then continues with the next statement. |
| lvl:control_flow@L3#2 \| level L3 way relation_to_neighbour: A flowchart draws control flow: each path through the diagram is a possible path of execution, with a decision box for each if or loop test. |
| lvl:control_flow@L3#3 \| level L3 way purpose_of_the_whole: Plan: control flow lets one program do different things for different inputs (selection) and do a thing many times (repetition), so a short program can specify a potentially unlimited number of executed statements. |
| lvl:control_flow@L4#0 \| level L4 way predict_a_change: Reordering two statements generally changes the result whenever one reads a variable that the other writes (e.g. swapping x = x + 1 and print(x)); reordering is safe only for statements that touch disjoint variables. Moving a statement into or out of a branch or loop body changes how many times it runs (0, 1 or n). |
| lvl:control_flow@L4#1 \| level L4 way transfer_to_a_new_program: Reason in terms of control flow whenever you must predict output or find why a statement did or did not run: trace which statement is reached next. Straight-line sequence suffices only when every input needs the same steps exactly once; otherwise add a conditional or a loop. |
| lvl:control_flow@L4#2 \| level L4 way diagnose_an_error: rule broken: A local variable must be definitely assigned on every control-flow path before it is read. Kind: compile-time. Trigger class: A variable assigned only inside an if branch (or only in a loop body) and read after it: the compiler reports 'variable might not have been initialized'. |
| lvl:control_flow@L4#3 \| level L4 way diagnose_an_error: rule broken: Statements run in the order written, so an update must come before the statement that needs the new value. Kind: silent logic error. Trigger class: Swapping two dependent statements (e.g. printing before updating, or incrementing the index before using it): output is shifted or off by one pass, with no error. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:control_flow@L0#0 (Score: 0.45)}
{lvl:control_flow@L2#1 (Score: 0.3)}
{lvl:control_flow@L0#3 (Score: 0.25)}
```
Kept by pruning: lvl:control_flow@L0#0 (0.45), lvl:control_flow@L2#1 (0.30), lvl:control_flow@L0#3 (0.25)

Label prompt for level way lvl:control_flow@L0#0 (the model saw only the concept, the way's name 'everyday reading' and its text; no level):
> Folk meaning: in everyday instructions a listener reads the whole list at once, does steps in a sensible order and infers what the writer wants. Java instead executes statements one at a time, top to bottom, each only when control reaches it, and nothing later in the program has any effect earlier.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:control_flow@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> The output is the sequence of print statements that control actually reaches, in the order reached: e.g. TenHellos prints 1st, 2nd, 3rd Hello once each, then 4th to 10th Hello from the loop.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.85
```

Label prompt for level way lvl:control_flow@L0#3 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the sequencing rule (LT) unevenly: e.g. runs a later line before an earlier one, lets an if that was already passed fire again later, or ends the program when a loop or a false if ends instead of continuing with the next statement. (here mixing in: "Statements in a program can run in any order")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:control_flow@L0#0 | not applied |  |  | 0.0 |  |  |
| lvl:control_flow@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:control_flow@L0#3 | not applied |  |  | 0.0 |  |  |

#### if_else_statement, misconceptions: 6 edges offered
Edges offered:
| edge |
|---|
| err:if_else_statement->ElseWithCondition \| wrong rule: Writes else (x > 0) with a condition but no if. \| right rule: else takes no condition; use else if (x > 0). |
| mis:if_else_statement->BothBranchesExecute \| wrong rule: Both the then branch and the else branch run. \| right rule: Exactly one of the two branches runs. |
| mis:if_else_statement->BranchChosenByIntent \| wrong rule: Which branch runs is decided by what the programmer meant or by code outside the if, not by whether the condition is true. \| right rule: The then branch runs exactly when the condition evaluates to true at the moment it is tested; otherwise the else branch (if any) runs. |
| mis:if_else_statement->ConditionalIsSequence \| wrong rule: if (c) A else B is the same as if (c) A followed by if (!c) B. \| right rule: In if-else, c is tested once and exactly one of A or B runs, even if A changes the variables that c uses. |
| mis:if_else_statement->DuplicateCodeInBranches \| wrong rule: Writes the same statements in both branches of an if-else. \| right rule: Code common to both branches belongs before or after the if-else. |
| mis:if_else_statement->ElseAlwaysExecutes \| wrong rule: The else branch of an if-else statement always executes. \| right rule: The else branch runs only when the if condition is false. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
mis:if_else_statement->BranchChosenByIntent (0.50), mis:if_else_statement->ElseAlwaysExecutes (0.30), mis:if_else_statement->BothBranchesExecute (0.20)
```
Kept by pruning: mis:if_else_statement->BranchChosenByIntent (0.50), mis:if_else_statement->ElseAlwaysExecutes (0.30), mis:if_else_statement->BothBranchesExecute (0.20)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:if_else_statement->BranchChosenByIntent \| wrong rule: Which branch runs is decided by what the programmer meant or by code outside the if, not by whether the condition is true. \| right rule: The then branch runs exactly when the condition evaluates to true at the moment it is tested; otherwise the else branch (if any) runs. |
| mis:if_else_statement->ElseAlwaysExecutes \| wrong rule: The else branch of an if-else statement always executes. \| right rule: The else branch runs only when the if condition is false. |
| mis:if_else_statement->BothBranchesExecute \| wrong rule: Both the then branch and the else branch run. \| right rule: Exactly one of the two branches runs. |
Model's reply:
```
mis:if_else_statement->BranchChosenByIntent | not applied | clauses: none | confidence: 0.9
mis:if_else_statement->ElseAlwaysExecutes | not applied | clauses: none | confidence: 0.9
mis:if_else_statement->BothBranchesExecute | not applied | clauses: none | confidence: 0.9
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:if_else_statement->BranchChosenByIntent | not applied |  |  | 0.0 |  |  |
| mis:if_else_statement->ElseAlwaysExecutes | not applied |  |  | 0.0 |  |  |
| mis:if_else_statement->BothBranchesExecute | not applied |  |  | 0.0 |  |  |

#### if_else_statement, level ways: 17 edges offered
Edges offered:
| edge |
|---|
| lvl:if_else_statement@L0#0 \| level L0 way everyday_reading: Folk meaning: in English 'otherwise' can be heard as 'and then also', and 'if ... else ...' as two separate rules that are each checked. In Java the condition is tested once and exactly one of the two branches runs. |
| lvl:if_else_statement@L0#5 \| level L0 way inconsistent_trace: Applies the if-else rule (LT) unevenly: on some traces runs both branches, re-tests the condition after the then branch to decide whether to run the else branch, or runs the else branch whatever the condition. (here mixing in: "if-else is the same as two separate ifs") |
| lvl:if_else_statement@L1#0 \| level L1 way names_the_element: The if-else statement is a two-way conditional: it executes one of two statements (or blocks) depending on a boolean condition. |
| lvl:if_else_statement@L1#1 \| level L1 way states_the_rule: Syntax: if (<boolean expression>) <statements T> else <statements F>; it executes one or the other, but not both: T if the condition is true, F otherwise. |
| lvl:if_else_statement@L2#0 \| level L2 way line_by_line_trace: Evaluate the condition once; if true execute the then branch, otherwise execute the else branch; then continue with the statement after the whole if-else (the condition is not re-tested). |
| lvl:if_else_statement@L2#1 \| level L2 way output_prediction: Exactly one branch's effects occur; e.g. if (x > y) max = x; else max = y; leaves max equal to the larger value, and the coin flip prints either Heads or Tails, never both. |
| lvl:if_else_statement@L2#2 \| level L2 way one_step_execution: One execution reads the condition's variables and applies the effects of exactly one branch; a change made by the then branch to the condition's variables cannot cause the else branch to run. |
| lvl:if_else_statement@L3#0 \| level L3 way relation_to_neighbour: if-else is an if with a second branch for the false case; dropping the else gives a plain if in which the false case does nothing. |
| lvl:if_else_statement@L3#1 \| level L3 way relation_to_neighbour: Either branch may itself be an if or if-else, which nests conditionals; without braces an inner else pairs with the innermost if. |
| lvl:if_else_statement@L3#2 \| level L3 way relation_to_neighbour: c ? a : b is the expression form of if (c) v = a; else v = b;: it yields a value of one branch, evaluating only the selected operand. |
| lvl:if_else_statement@L3#3 \| level L3 way relation_to_neighbour: An else-if chain is an if-else whose else part is another if-else, repeated; the final else catches all remaining cases. |
| lvl:if_else_statement@L3#4 \| level L3 way purpose_of_the_whole: Plan: two-way choice / alternative action - compute one of two results (max of two values, heads or tails, error message or quotient). |
| lvl:if_else_statement@L4#0 \| level L4 way predict_a_change: Swapping the two branches and negating the condition preserves behaviour; swapping the branches alone inverts it. Replacing if-else with two separate ifs is not equivalent when the then branch changes a variable used in the condition. |
| lvl:if_else_statement@L4#1 \| level L4 way diagnose_an_error: rule broken: else takes no condition: it is the branch for the false case of the preceding if; a further test needs else if (...). Kind: compile-time. Trigger class: Writing else (x > 0) stmt; the compiler reads (x > 0) as the start of an expression statement, which is not a valid statement, and reports an error. |
| lvl:if_else_statement@L4#2 \| level L4 way transfer_to_a_new_program: Use if-else when exactly one of two actions must happen; use a plain if when nothing happens in the false case, an else-if chain or switch for more than two mutually exclusive cases, and c ? a : b when you only need to choose a value. |
| lvl:if_else_statement@L4#3 \| level L4 way diagnose_an_error: rule broken: Code common to both branches belongs before or after the if-else. Kind: silent logic error. Trigger class: A statement duplicated in both branches and later edited in only one: the two cases silently diverge. |
| lvl:if_else_statement@L4#4 \| level L4 way diagnose_an_error: rule broken: An else must directly follow the then statement of an if. Kind: compile-time. Trigger class: if (c) x = 1; y = 2; else x = 0;: the else has no matching if ('else without if'). |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:if_else_statement@L2#1 (Score: 0.5)}
{lvl:if_else_statement@L2#0 (Score: 0.3)}
{lvl:if_else_statement@L0#5 (Score: 0.2)}
```
Kept by pruning: lvl:if_else_statement@L2#1 (0.50), lvl:if_else_statement@L2#0 (0.30), lvl:if_else_statement@L0#5 (0.20)

Label prompt for level way lvl:if_else_statement@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> Exactly one branch's effects occur; e.g. if (x > y) max = x; else max = y; leaves max equal to the larger value, and the coin flip prints either Heads or Tails, never both.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:if_else_statement@L2#0 (the model saw only the concept, the way's name 'line by line trace' and its text; no level):
> Evaluate the condition once; if true execute the then branch, otherwise execute the else branch; then continue with the statement after the whole if-else (the condition is not re-tested).

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:if_else_statement@L0#5 (the model saw only the concept, the way's name 'inconsistent trace' and its text; no level):
> Applies the if-else rule (LT) unevenly: on some traces runs both branches, re-tests the condition after the then branch to decide whether to run the else branch, or runs the else branch whatever the condition. (here mixing in: "if-else is the same as two separate ifs")

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:if_else_statement@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:if_else_statement@L2#0 | not applied |  |  | 0.0 |  |  |
| lvl:if_else_statement@L0#5 | not applied |  |  | 0.0 |  |  |

**Kept on a path at this depth** (best N in tiers: held misconception / broken edge, then held error, then correct / confirmed / shown):

pre:string_literal->println [broken, tier 0, 0.39]

reached across pre:string_literal->println, entered next: string_literal

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | pre:string_literal->println |
| c2 | The cast makes it a double division | not covered yet |
| c3 | so you get the decimal part | not covered yet |

**Graph check, not yet read (the walk may not stop while these remain):** own edges of string_literal, par:assignment~comparison_operators, pre:expression->comparison_operators, pre:statement->control_flow

**Figure 6 (the graph after depth 3)**

![Figure 6](figure06.png)
### Depth 4: current concepts 'string_literal'

#### string_literal, misconceptions: 4 edges offered
Edges offered:
| edge |
|---|
| err:string_literal->SingleQuotedString \| wrong rule: Writes a string literal in single quotes: 'Hello'. \| right rule: String literals use double quotes; single quotes are for one char. |
| err:string_literal->StringLiteralSpansLines \| wrong rule: Breaks a string literal across two lines of code. \| right rule: String literals must not span more than one line of a program. |
| err:string_literal->UnclosedStringLiteral \| wrong rule: A string literal can be left without its closing quote. \| right rule: A String literal is enclosed in a pair of double quotes. |
| mis:string_literal->PrintedTextIsExecuted \| wrong rule: System.out.println("x is 5"); sets x to 5 (text inside the quotes is run as code). \| right rule: Characters inside double quotes are printed literally; println only displays them and changes no variable. Only an assignment such as x = 5; sets x. |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{mis:string_literal->PrintedTextIsExecuted (Score: 0.8)}
{err:string_literal->UnclosedStringLiteral (Score: 0.1)}
{err:string_literal->SingleQuotedString (Score: 0.1)}
```
Kept by pruning: mis:string_literal->PrintedTextIsExecuted (0.80), err:string_literal->UnclosedStringLiteral (0.10), err:string_literal->SingleQuotedString (0.10)

Label prompt for the misconception / error edges (judged together). The model saw the program, the question, the numbered clauses, and these edges with their wrong rule and right rule:
| edge |
|---|
| mis:string_literal->PrintedTextIsExecuted \| wrong rule: System.out.println("x is 5"); sets x to 5 (text inside the quotes is run as code). \| right rule: Characters inside double quotes are printed literally; println only displays them and changes no variable. Only an assignment such as x = 5; sets x. |
| err:string_literal->UnclosedStringLiteral \| wrong rule: A string literal can be left without its closing quote. \| right rule: A String literal is enclosed in a pair of double quotes. |
| err:string_literal->SingleQuotedString \| wrong rule: Writes a string literal in single quotes: 'Hello'. \| right rule: String literals use double quotes; single quotes are for one char. |
Model's reply:
```
mis:string_literal->PrintedTextIsExecuted | not applied | clauses: none | confidence: 0.9
err:string_literal->UnclosedStringLiteral | not applied | clauses: none | confidence: 0.95
err:string_literal->SingleQuotedString | not applied | clauses: none | confidence: 0.95
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| mis:string_literal->PrintedTextIsExecuted | not applied |  |  | 0.0 |  |  |
| err:string_literal->UnclosedStringLiteral | not applied |  |  | 0.0 |  |  |
| err:string_literal->SingleQuotedString | not applied |  |  | 0.0 |  |  |

#### string_literal, level ways: 16 edges offered
Edges offered:
| edge |
|---|
| lvl:string_literal@L0#0 \| level L0 way everyday_reading: Everyday reading: quotation marks are just decoration, and spacing inside them is tidied up like ordinary text; single and double quotes mean the same thing. Java rule: characters within double quotes are taken literally (every space counts), the quotes are not part of the value, and single quotes are for one char. |
| lvl:string_literal@L1#0 \| level L1 way names_the_element: A string literal is a sequence of characters written in double quotes in the source code, e.g. "Hello, World"; it is a value of type String. |
| lvl:string_literal@L1#1 \| level L1 way states_the_rule: A String literal is enclosed in a pair of double quotes; everything inside is taken literally, special characters such as " and newline are written with escape sequences, and the literal must close on the same source line. |
| lvl:string_literal@L2#0 \| level L2 way line_by_line_trace: The literal is a constant: it is fixed at compile time and simply supplies its value when the expression containing it is evaluated. |
| lvl:string_literal@L2#1 \| level L2 way output_prediction: Value is the String of the characters between the quotes (escape sequences replaced by the characters they stand for), without the quotes: "Hi, " has 4 characters, and printing it shows Hi, followed by a space. |
| lvl:string_literal@L2#2 \| level L2 way one_step_execution: Evaluating a literal changes no state; it just yields the same String value every time. |
| lvl:string_literal@L3#0 \| level L3 way relation_to_neighbour: print("s") outputs exactly the characters between the quotes, without the quotes. |
| lvl:string_literal@L3#1 \| level L3 way relation_to_neighbour: println("s") outputs exactly the characters between the quotes and then ends the line. |
| lvl:string_literal@L3#2 \| level L3 way relation_to_neighbour: Escape sequences (\n, \t, \", \\) are written inside string or char literals to stand for one character that cannot be typed directly. |
| lvl:string_literal@L3#3 \| level L3 way purpose_of_the_whole: Purpose: state fixed text exactly, such as messages, labels and prompts, so you can precisely specify what gets printed. |
| lvl:string_literal@L4#0 \| level L4 way predict_a_change: Changing any character inside the quotes (including adding a space) changes the output by exactly that character; changing spaces outside the quotes changes nothing. |
| lvl:string_literal@L4#1 \| level L4 way diagnose_an_error: rule broken: A String literal is enclosed in a pair of double quotes. Kind: compile-time. Trigger class: a missing closing (or opening) quote; the compiler may get very confused and report errors elsewhere |
| lvl:string_literal@L4#2 \| level L4 way diagnose_an_error: rule broken: String literals use double quotes; single quotes are for one char. Kind: compile-time. Trigger class: 'Hello' (more than one character in single quotes): unclosed character literal |
| lvl:string_literal@L4#3 \| level L4 way diagnose_an_error: rule broken: A string literal must not span source lines. Kind: compile-time. Trigger class: a line break inside "...": unclosed string literal |
| lvl:string_literal@L4#4 \| level L4 way transfer_to_a_new_program: Use a string literal for fixed text known when the program is written; use args, input or variables for text that varies between runs, and char literals for a single character. |
| lvl:string_literal@L4#5 \| level L4 way diagnose_an_error: rule broken: A double quote inside a string must be escaped. Kind: compile-time. Trigger class: "She said "Hi"" without \" |

Prune: the model's reply (edge id and score; the kept edges sum to 1):
```
{lvl:string_literal@L2#1 (Score: 0.45)}
{lvl:string_literal@L3#1 (Score: 0.35)}
{lvl:string_literal@L3#3 (Score: 0.2)}
```
Kept by pruning: lvl:string_literal@L2#1 (0.45), lvl:string_literal@L3#1 (0.35), lvl:string_literal@L3#3 (0.20)

Label prompt for level way lvl:string_literal@L2#1 (the model saw only the concept, the way's name 'output prediction' and its text; no level):
> Value is the String of the characters between the quotes (escape sequences replaced by the characters they stand for), without the quotes: "Hi, " has 4 characters, and printing it shows Hi, followed by a space.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.95
```

Label prompt for level way lvl:string_literal@L3#1 (the model saw only the concept, the way's name 'relation to neighbour' and its text; no level):
> println("s") outputs exactly the characters between the quotes and then ends the line.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.97
```

Label prompt for level way lvl:string_literal@L3#3 (the model saw only the concept, the way's name 'purpose of the whole' and its text; no level):
> Purpose: state fixed text exactly, such as messages, labels and prompts, so you can precisely specify what gets printed.

Model's reply:
```
Label: not shown
Clauses: none
Confidence: 0.97
```

Parsed labels:
| edge | label | prune score | confidence | score | clauses | words |
|---|---|---|---|---|---|---|
| lvl:string_literal@L2#1 | not applied |  |  | 0.0 |  |  |
| lvl:string_literal@L3#1 | not applied |  |  | 0.0 |  |  |
| lvl:string_literal@L3#3 | not applied |  |  | 0.0 |  |  |

**Kept on a path at this depth:** nothing

**Clause coverage after this depth:**
| clause | text | covered by |
|---|---|---|
| c1 | 3.5 | pre:string_literal->println |
| c2 | The cast makes it a double division | not covered yet |
| c3 | so you get the decimal part | not covered yet |

**Graph check, not yet read (the walk may not stop while these remain):** par:assignment~comparison_operators, pre:expression->comparison_operators, pre:statement->control_flow

**Figure 7 (the graph after depth 4)**

![Figure 7](figure07.png)


## 4. Diagnosis (rules over the labelled graph, no model)
Root-cause walk: held on [], broken at ['println'], prerequisites {'expression': ['arithmetic_operators'], 'if_statement': ['boolean_expression', 'control_flow'], 'boolean_expression': ['comparison_operators'], 'floating_point_precision': ['double'], 'comparison_operators': ['expression'], 'if_else_statement': ['if_statement'], 'control_flow': ['statement'], 'println': ['statement', 'string_literal']}
- arithmetic_operators: L4 via predict_a_change "The cast makes it a double division; so you get the decimal part"
- double: L4 via diagnose_an_error "The cast makes it a double division; so you get the decimal part"

**Stop:** max depth at depth 4; prompts {'prune': 25, 'label': 48}, fresh model calls 0, cache hits 73

### Held misconceptions
(none)

### Broken concept edges
| edge | quote | score | consequence_of |
|---|---|---|---|
| pre:string_literal->println | 3.5 | 0.385 |  |

### Root cause: none

### Level per concept
| concept | level | way | misconception | source | quote | rule | also | note |
|---|---|---|---|---|---|---|---|---|
| arithmetic_operators | L4 | predict_a_change |  | level_way | The cast makes it a double division; so you get the decimal part |  | L2 output_prediction, L1 states_the_rule |  |
| double | L4 | diagnose_an_error |  | level_way | The cast makes it a double division; so you get the decimal part |  | L3 relation_to_neighbour, L1 states_the_rule |  |

### Overall level: L4

Clause coverage (final):
| clause | covered by |
|---|---|
| 3.5 | lvl:arithmetic_operators@L2#1 |
| The cast makes it a double division | lvl:double@L3#0 |
| so you get the decimal part | lvl:double@L3#0 |

**Figure 8 (final picture)**

![Figure 8](figure08.png)


## 5. Every labelled edge (the evidence list, kept or not)
| depth | source | edge | kind | label | prune_score | confidence | score | clauses | quote |
|---|---|---|---|---|---|---|---|---|---|
| 1 | double | lvl:double@L3#0 | level | shown | 0.5 | 0.7 | 0.35 | [2, 3] | The cast makes it a double division; so you get the decimal part |
| 1 | double | lvl:double@L1#1 | level | shown | 0.3 | 0.7 | 0.21 | [2, 3] | The cast makes it a double division; so you get the decimal part |
| 1 | double | lvl:double@L4#3 | level | shown | 0.2 | 0.6 | 0.12 | [2, 3] | The cast makes it a double division; so you get the decimal part |
| 1 | arithmetic_operators | lvl:arithmetic_operators@L4#0 | level | shown | 0.45 | 0.75 | 0.3375 | [2, 3] | The cast makes it a double division; so you get the decimal part |
| 1 | arithmetic_operators | lvl:arithmetic_operators@L2#1 | level | shown | 0.35 | 0.6 | 0.21 | [1, 2, 3] | 3.5; The cast makes it a double division; so you get the decimal part |
| 1 | arithmetic_operators | lvl:arithmetic_operators@L1#1 | level | shown | 0.2 | 0.55 | 0.11 | [2, 3] | The cast makes it a double division; so you get the decimal part |
| 3 | println | pre:string_literal->println | prerequisite | broken | 0.7 | 0.55 | 0.385 | [1] | 3.5 |


## 6. The paths (what the keep step kept, depth by depth)
| depth | source | edge | kind | label | score | clauses |
|---|---|---|---|---|---|---|
| 1 | double | lvl:double@L3#0 | level | shown | 0.35 | [2, 3] |
| 1 | arithmetic_operators | lvl:arithmetic_operators@L4#0 | level | shown | 0.3375 | [2, 3] |
| 1 | double | lvl:double@L1#1 | level | shown | 0.21 | [2, 3] |
| 3 | println | pre:string_literal->println | prerequisite | broken | 0.385 | [1] |


## 7. Raw traversal log
```
clauses: 3.5 | The cast makes it a double division | so you get the decimal part
start: generic concepts whose tagging text occurs in the answer: double
start: BM25 of the answer against each tagged concept: double 11.19 (hits 3), arithmetic_operators 6.38 (hits 1), if_statement 6.16 (hits 1), println 5.40 (hits 1), boolean_expression 4.54 (hits 1), floating_point_precision 3.23 (hits 3), control_flow 2.61 (hits 1), comparison_operators 1.98 (hits 1), if_else_statement 1.71 (hits 1)
start: chosen double, arithmetic_operators, if_statement
start concepts (3): double, arithmetic_operators, if_statement
--- depth 1: current concepts ['double', 'arithmetic_operators', 'if_statement']
  double (misconceptions): 2 candidates -> kept mis:double->WholeNumberLiteralIsInt (0.60), mis:double->FloatingPointIsExact (0.40)
    mis:double->WholeNumberLiteralIsInt -> not applied (0.00) ""
    mis:double->FloatingPointIsExact -> not applied (0.00) ""
  double (level ways): 19 candidates -> kept lvl:double@L3#0 (0.50), lvl:double@L1#1 (0.30), lvl:double@L4#3 (0.20)
    lvl:double@L3#0 -> shown (0.35) "The cast makes it a double division; so you get the decimal part"
    lvl:double@L1#1 -> shown (0.21) "The cast makes it a double division; so you get the decimal part"
    lvl:double@L4#3 -> shown (0.12) "The cast makes it a double division; so you get the decimal part"
  arithmetic_operators (misconceptions): 3 candidates -> kept 
  arithmetic_operators (level ways): 18 candidates -> kept lvl:arithmetic_operators@L4#0 (0.45), lvl:arithmetic_operators@L2#1 (0.35), lvl:arithmetic_operators@L1#1 (0.20)
    lvl:arithmetic_operators@L4#0 -> shown (0.34) "The cast makes it a double division; so you get the decimal part"
    lvl:arithmetic_operators@L2#1 -> shown (0.21) "3.5; The cast makes it a double division; so you get the decimal part"
    lvl:arithmetic_operators@L1#1 -> shown (0.11) "The cast makes it a double division; so you get the decimal part"
  if_statement (misconceptions): 22 candidates -> kept mis:if_statement->BranchChosenByIntent (0.50), mis:if_statement->ThenAlwaysExecutes (0.30), mis:if_statement->ProgramLooksAhead (0.20)
    mis:if_statement->BranchChosenByIntent -> not applied (0.00) ""
    mis:if_statement->ThenAlwaysExecutes -> not applied (0.00) ""
    mis:if_statement->ProgramLooksAhead -> not applied (0.00) ""
  if_statement (level ways): 19 candidates -> kept lvl:if_statement@L2#1 (0.40), lvl:if_statement@L3#1 (0.35), lvl:if_statement@L0#14 (0.25)
    lvl:if_statement@L2#1 -> not applied (0.00) ""
    lvl:if_statement@L3#1 -> not applied (0.00) ""
    lvl:if_statement@L0#14 -> not applied (0.00) ""
  kept: lvl:double@L3#0 [shown, tier 2, 0.35], lvl:arithmetic_operators@L4#0 [shown, tier 2, 0.34], lvl:double@L1#1 [shown, tier 2, 0.21]
  clause coverage: c1 uncovered; c2 uncovered; c3 uncovered
  uncovered clauses bring in: println (5.41), boolean_expression (4.02), floating_point_precision (3.16)
  not yet read: own edges of boolean_expression, own edges of floating_point_precision, own edges of println, pre:arithmetic_operators->expression, pre:boolean_expression->if_statement, pre:control_flow->if_statement, pre:double->floating_point_precision, pre:if_statement->if_else_statement
--- depth 2: current concepts ['double', 'arithmetic_operators', 'if_statement', 'println', 'boolean_expression', 'floating_point_precision']
  double (concept edges): 1 candidates -> kept pre:double->floating_point_precision (1.00)
    pre:double->floating_point_precision -> not applied (0.00) ""
  arithmetic_operators (concept edges): 1 candidates -> kept pre:arithmetic_operators->expression (1.00)
    pre:arithmetic_operators->expression -> not applied (0.00) ""
  if_statement (concept edges): 3 candidates -> kept pre:if_statement->if_else_statement (0.50), pre:boolean_expression->if_statement (0.30), pre:control_flow->if_statement (0.20)
    pre:if_statement->if_else_statement -> not applied (0.00) ""
    pre:boolean_expression->if_statement -> not applied (0.00) ""
    pre:control_flow->if_statement -> not applied (0.00) ""
  println (misconceptions): 6 candidates -> kept mis:println->PrintedTextIsExecuted (0.60), mis:println->PrintNewLineFirst (0.20), mis:println->PrintEqualsPrintln (0.20)
    mis:println->PrintedTextIsExecuted -> not applied (0.00) ""
    mis:println->PrintNewLineFirst -> not applied (0.00) ""
    mis:println->PrintEqualsPrintln -> not applied (0.00) ""
  println (level ways): 17 candidates -> kept lvl:println@L2#1 (0.40), lvl:println@L3#1 (0.35), lvl:println@L3#3 (0.25)
    lvl:println@L2#1 -> not applied (0.00) ""
    lvl:println@L3#1 -> not applied (0.00) ""
    lvl:println@L3#3 -> not applied (0.00) ""
  boolean_expression (misconceptions): 8 candidates -> kept mis:boolean_expression->ArithmeticInsteadOfBoolean (0.80), mis:boolean_expression->MapToBooleanWithIf (0.10), mis:boolean_expression->ComparisonWithBooleanLiteral (0.10)
    mis:boolean_expression->ArithmeticInsteadOfBoolean -> not applied (0.00) ""
    mis:boolean_expression->MapToBooleanWithIf -> not applied (0.00) ""
    mis:boolean_expression->ComparisonWithBooleanLiteral -> not applied (0.00) ""
  boolean_expression (level ways): 19 candidates -> kept lvl:boolean_expression@L2#1 (0.50), lvl:boolean_expression@L3#2 (0.30), lvl:boolean_expression@L0#5 (0.20)
    lvl:boolean_expression@L2#1 -> not applied (0.00) ""
    lvl:boolean_expression@L3#2 -> not applied (0.00) ""
    lvl:boolean_expression@L0#5 -> not applied (0.00) ""
  floating_point_precision (misconceptions): 1 candidates -> kept mis:floating_point_precision->FloatingPointIsExact (1.00)
    mis:floating_point_precision->FloatingPointIsExact -> not applied (0.00) ""
  floating_point_precision (level ways): 14 candidates -> kept lvl:floating_point_precision@L0#0 (0.50), lvl:floating_point_precision@L2#1 (0.30), lvl:floating_point_precision@L0#2 (0.20)
    lvl:floating_point_precision@L0#0 -> not applied (0.00) ""
    lvl:floating_point_precision@L2#1 -> not applied (0.00) ""
    lvl:floating_point_precision@L0#2 -> not applied (0.00) ""
  nothing kept at this depth
  clause coverage: c1 uncovered; c2 uncovered; c3 uncovered
  uncovered clauses bring in: comparison_operators (2.85), control_flow (2.80), if_else_statement (1.78)
  not yet read: own edges of comparison_operators, own edges of control_flow, own edges of if_else_statement, pre:comparison_operators->boolean_expression, pre:statement->println, pre:string_literal->println
--- depth 3: current concepts ['println', 'boolean_expression', 'comparison_operators', 'control_flow', 'if_else_statement']
  println (concept edges): 2 candidates -> kept pre:string_literal->println (0.70), pre:statement->println (0.30)
    pre:string_literal->println -> broken (0.39) "3.5"
    pre:statement->println -> not applied (0.00) ""
  boolean_expression (concept edges): 2 candidates -> kept pre:boolean_expression->if_statement (0.60), pre:comparison_operators->boolean_expression (0.40)
    pre:boolean_expression->if_statement -> not applied (0.00) ""
    pre:comparison_operators->boolean_expression -> not applied (0.00) ""
  comparison_operators (misconceptions): 8 candidates -> kept mis:comparison_operators->EqualityOperatorComparesObjectsValues (0.40), mis:comparison_operators->AssignCompares (0.30), mis:comparison_operators->EqualsComparesReferences (0.30)
    mis:comparison_operators->EqualityOperatorComparesObjectsValues -> not applied (0.00) ""
    mis:comparison_operators->AssignCompares -> not applied (0.00) ""
    mis:comparison_operators->EqualsComparesReferences -> not applied (0.00) ""
  comparison_operators (level ways): 20 candidates -> kept lvl:comparison_operators@L4#6 (0.40), lvl:comparison_operators@L2#0 (0.35), lvl:comparison_operators@L2#1 (0.25)
    lvl:comparison_operators@L4#6 -> not applied (0.00) ""
    lvl:comparison_operators@L2#0 -> not applied (0.00) ""
    lvl:comparison_operators@L2#1 -> not applied (0.00) ""
  control_flow (misconceptions): 11 candidates -> kept mis:control_flow->StatementsRunInAnyOrder (0.40), mis:control_flow->CodeAfterIfSkipped (0.30), mis:control_flow->FalseConditionRestartsProgram (0.30)
    mis:control_flow->StatementsRunInAnyOrder -> not applied (0.00) ""
    mis:control_flow->CodeAfterIfSkipped -> not applied (0.00) ""
    mis:control_flow->FalseConditionRestartsProgram -> not applied (0.00) ""
  control_flow (level ways): 15 candidates -> kept lvl:control_flow@L0#0 (0.45), lvl:control_flow@L2#1 (0.30), lvl:control_flow@L0#3 (0.25)
    lvl:control_flow@L0#0 -> not applied (0.00) ""
    lvl:control_flow@L2#1 -> not applied (0.00) ""
    lvl:control_flow@L0#3 -> not applied (0.00) ""
  if_else_statement (misconceptions): 6 candidates -> kept mis:if_else_statement->BranchChosenByIntent (0.50), mis:if_else_statement->ElseAlwaysExecutes (0.30), mis:if_else_statement->BothBranchesExecute (0.20)
    mis:if_else_statement->BranchChosenByIntent -> not applied (0.00) ""
    mis:if_else_statement->ElseAlwaysExecutes -> not applied (0.00) ""
    mis:if_else_statement->BothBranchesExecute -> not applied (0.00) ""
  if_else_statement (level ways): 17 candidates -> kept lvl:if_else_statement@L2#1 (0.50), lvl:if_else_statement@L2#0 (0.30), lvl:if_else_statement@L0#5 (0.20)
    lvl:if_else_statement@L2#1 -> not applied (0.00) ""
    lvl:if_else_statement@L2#0 -> not applied (0.00) ""
    lvl:if_else_statement@L0#5 -> not applied (0.00) ""
  kept: pre:string_literal->println [broken, tier 0, 0.39]
  reached across pre:string_literal->println, entered next: string_literal
  clause coverage: c1 <- pre:string_literal->println; c2 uncovered; c3 uncovered
  not yet read: own edges of string_literal, par:assignment~comparison_operators, pre:expression->comparison_operators, pre:statement->control_flow
--- depth 4: current concepts ['string_literal'] (closing round: own edges only)
  string_literal (misconceptions): 4 candidates -> kept mis:string_literal->PrintedTextIsExecuted (0.80), err:string_literal->UnclosedStringLiteral (0.10), err:string_literal->SingleQuotedString (0.10)
    mis:string_literal->PrintedTextIsExecuted -> not applied (0.00) ""
    err:string_literal->UnclosedStringLiteral -> not applied (0.00) ""
    err:string_literal->SingleQuotedString -> not applied (0.00) ""
  string_literal (level ways): 16 candidates -> kept lvl:string_literal@L2#1 (0.45), lvl:string_literal@L3#1 (0.35), lvl:string_literal@L3#3 (0.20)
    lvl:string_literal@L2#1 -> not applied (0.00) ""
    lvl:string_literal@L3#1 -> not applied (0.00) ""
    lvl:string_literal@L3#3 -> not applied (0.00) ""
  nothing kept at this depth
  clause coverage: c1 <- pre:string_literal->println; c2 uncovered; c3 uncovered
  not yet read: par:assignment~comparison_operators, pre:expression->comparison_operators, pre:statement->control_flow
  root-cause walk: held on [], broken at ['println'], prerequisites {'expression': ['arithmetic_operators'], 'if_statement': ['boolean_expression', 'control_flow'], 'boolean_expression': ['comparison_operators'], 'floating_point_precision': ['double'], 'comparison_operators': ['expression'], 'if_else_statement': ['if_statement'], 'control_flow': ['statement'], 'println': ['statement', 'string_literal']}
  level arithmetic_operators: L4 via predict_a_change "The cast makes it a double division; so you get the decimal part"
  level double: L4 via diagnose_an_error "The cast makes it a double division; so you get the decimal part"
```


## 8. Full diagnosis JSON
```json
{
 "scenario": "MY29",
 "settings": {
  "width": 3,
  "depth": 3,
  "prune_tools": "llm",
  "model": "claude-opus-5-5",
  "backend": "claude-cli"
 },
 "clauses": [
  "3.5",
  "The cast makes it a double division",
  "so you get the decimal part"
 ],
 "start_concepts": [
  "double",
  "arithmetic_operators",
  "if_statement"
 ],
 "held_misconceptions": [],
 "error_evidence": [],
 "shadowed": [],
 "broken_edges": [
  {
   "edge": "pre:string_literal->println",
   "quote": "3.5",
   "score": 0.385
  }
 ],
 "root_cause": null,
 "levels": {
  "arithmetic_operators": {
   "level": "L4",
   "way": "predict_a_change",
   "quote": "The cast makes it a double division; so you get the decimal part",
   "edge": "lvl:arithmetic_operators@L4#0",
   "source": "level_way",
   "also_shown": [
    "L2 output_prediction",
    "L1 states_the_rule"
   ]
  },
  "double": {
   "level": "L4",
   "way": "diagnose_an_error",
   "quote": "The cast makes it a double division; so you get the decimal part",
   "edge": "lvl:double@L4#3",
   "source": "level_way",
   "also_shown": [
    "L3 relation_to_neighbour",
    "L1 states_the_rule"
   ]
  }
 },
 "overall_level": "L4",
 "paths": [
  {
   "depth": 1,
   "source": "double",
   "edge": "lvl:double@L3#0",
   "target": "double@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.7,
   "prune_score": 0.5,
   "score": 0.35,
   "logic": "Type flow decides the division: with two int operands a / b discards the fraction (10/3 is 3, and double d = 10/3; stores 3.0); if either operand is double the other is promoted and the fraction is kept (10/3.0 is 3.3333333333333335).",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  },
  {
   "depth": 1,
   "source": "arithmetic_operators",
   "edge": "lvl:arithmetic_operators@L4#0",
   "target": "arithmetic_operators@L4",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.75,
   "prune_score": 0.45,
   "score": 0.3375,
   "logic": "Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power.",
   "twin_logic": null,
   "level": "L4",
   "way": "predict_a_change"
  },
  {
   "depth": 1,
   "source": "double",
   "edge": "lvl:double@L1#1",
   "target": "double@L1",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.7,
   "prune_score": 0.3,
   "score": 0.21,
   "logic": "Values are 64-bit IEEE 754 numbers (about 15 significant digits) plus the special values Infinity and NaN; +, -, *, / (and %) are defined; if at least one operand of a binary arithmetic operator is double, the operation is a floating-point operation.",
   "twin_logic": null,
   "level": "L1",
   "way": "states_the_rule"
  },
  {
   "depth": 3,
   "source": "println",
   "edge": "pre:string_literal->println",
   "target": "string_literal",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1
   ],
   "quote": "3.5",
   "confidence": 0.55,
   "prune_score": 0.7,
   "score": 0.385,
   "logic": "println(\"s\") outputs exactly the characters between the quotes and then ends the line.",
   "twin_logic": null
  }
 ],
 "evidence": [
  {
   "depth": 1,
   "source": "double",
   "edge": "lvl:double@L3#0",
   "target": "double@L3",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.7,
   "prune_score": 0.5,
   "score": 0.35,
   "logic": "Type flow decides the division: with two int operands a / b discards the fraction (10/3 is 3, and double d = 10/3; stores 3.0); if either operand is double the other is promoted and the fraction is kept (10/3.0 is 3.3333333333333335).",
   "twin_logic": null,
   "level": "L3",
   "way": "relation_to_neighbour"
  },
  {
   "depth": 1,
   "source": "double",
   "edge": "lvl:double@L1#1",
   "target": "double@L1",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.7,
   "prune_score": 0.3,
   "score": 0.21,
   "logic": "Values are 64-bit IEEE 754 numbers (about 15 significant digits) plus the special values Infinity and NaN; +, -, *, / (and %) are defined; if at least one operand of a binary arithmetic operator is double, the operation is a floating-point operation.",
   "twin_logic": null,
   "level": "L1",
   "way": "states_the_rule"
  },
  {
   "depth": 1,
   "source": "double",
   "edge": "lvl:double@L4#3",
   "target": "double@L4",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.6,
   "prune_score": 0.2,
   "score": 0.12,
   "logic": "rule broken: Floating-point division needs at least one double operand. Kind: silent logic error. Trigger class: A formula assigned to a double whose division has only int operands, e.g. double y = 1 / 3; (gives 0.0) or (9/5)*c + 32.",
   "twin_logic": null,
   "level": "L4",
   "way": "diagnose_an_error"
  },
  {
   "depth": 1,
   "source": "arithmetic_operators",
   "edge": "lvl:arithmetic_operators@L4#0",
   "target": "arithmetic_operators@L4",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.75,
   "prune_score": 0.45,
   "score": 0.3375,
   "logic": "Changing an operand's type changes the operator's meaning (int / versus double /). Reordering * and / changes integer results (59 * 100 / 60 is 98, 59 / 60 * 100 is 0). Replacing ^ with Math.pow gives a real power.",
   "twin_logic": null,
   "level": "L4",
   "way": "predict_a_change"
  },
  {
   "depth": 1,
   "source": "arithmetic_operators",
   "edge": "lvl:arithmetic_operators@L2#1",
   "target": "arithmetic_operators@L2",
   "kind": "level",
   "label": "shown",
   "clauses": [
    1,
    2,
    3
   ],
   "quote": "3.5; The cast makes it a double division; so you get the decimal part",
   "confidence": 0.6,
   "prune_score": 0.35,
   "score": 0.21,
   "logic": "With ints, 17 / 3 is 5 and 17 % 3 is 2. With doubles, 5.0 / 3.0 is 1.6666666666666667. With mixed operands, the int is promoted first (10 / 3.0 is 3.333...).",
   "twin_logic": null,
   "level": "L2",
   "way": "output_prediction"
  },
  {
   "depth": 1,
   "source": "arithmetic_operators",
   "edge": "lvl:arithmetic_operators@L1#1",
   "target": "arithmetic_operators@L1",
   "kind": "level",
   "label": "shown",
   "clauses": [
    2,
    3
   ],
   "quote": "The cast makes it a double division; so you get the decimal part",
   "confidence": 0.55,
   "prune_score": 0.2,
   "score": 0.11,
   "logic": "Both operands must be numeric, and the result has the promoted type (int, long or double). * / % bind tighter than + -, all group left to right, and int / or % by zero throws ArithmeticException. Unary minus negates its operand (-x) and unary plus leaves it unchanged; they bind more tightly than * and /, so -a * b is (-a) * b.",
   "twin_logic": null,
   "level": "L1",
   "way": "states_the_rule"
  },
  {
   "depth": 3,
   "source": "println",
   "edge": "pre:string_literal->println",
   "target": "string_literal",
   "kind": "prerequisite",
   "label": "broken",
   "clauses": [
    1
   ],
   "quote": "3.5",
   "confidence": 0.55,
   "prune_score": 0.7,
   "score": 0.385,
   "logic": "println(\"s\") outputs exactly the characters between the quotes and then ends the line.",
   "twin_logic": null
  }
 ],
 "clause_coverage": {
  "3.5": "lvl:arithmetic_operators@L2#1",
  "The cast makes it a double division": "lvl:double@L3#0",
  "so you get the decimal part": "lvl:double@L3#0"
 },
 "unexplained_clauses": [],
 "stop_reason": "max depth",
 "depth_reached": 4,
 "llm_calls": 0,
 "llm_cache_hits": 73,
 "llm_calls_by_step": {
  "prune": 25,
  "label": 48
 }
}
```
